# The Impact of M&A on Inventor Mobility and Innovation
## Draft notebook for the GitHub package

This notebook is a narrative companion to the construction and analysis pipeline in the repository. It is written to do four jobs at once:

1. explain how the data are built and how the main empirical designs work,
2. present a disciplined headline story,
3. distinguish robust findings from fragile or method-dependent ones,
4. serve as a base for a sequence of public-facing writeups such as LinkedIn posts.

## Current headline story

The cleanest story is **not** that mergers and acquisitions uniformly reduce innovation output.  
The stronger and more defensible claim is narrower:

> **M&A appears to change inventor mobility and the direction of inventive search more clearly than it changes aggregate output, with especially important effects on the acquiror side.**

In particular, the most interesting pattern is that **acquiror-side inventor mobility rises after deals, and exploration becomes more prominent**. By contrast, effects on patents, citations, and related output measures are often more fragile across methods or complicated by pre-trends.

## What this notebook treats as primary versus secondary

### Primary findings
- Acquiror-side post-deal inventor mobility.
- Acquiror-side exploration outcomes.
- Heterogeneity by firm size, especially where negative baseline effects weaken among larger firms.

### Secondary findings
- Aggregate output measures such as patent counts, citations, and quality proxies.
- Arrivals and departures as supporting evidence for organizational reshuffling.
- Target-side effects, unless they are unusually stable across methods.

## Important caution

This draft reflects the current state of the project and the review notes. Some statements below are intentionally cautious because a few results are sensitive to estimator choice, event-study shape, or pre-trend concerns.

## Why the construction itself strengthens the project

For recruiting purposes, one of the strongest features of the project is that it is **not only a set of regressions**. It is a full research pipeline that builds analysis-ready firm and inventor panels from raw patent, accounting, and M&A data, and the final inventor event-study panel passes several useful integrity checks. In the completed construction run, the inventor M&A event-study panel is fully balanced over the `[-5, +5]` window for all retained inventor-events, with stable treated and control row counts at each relative year. Pre-period balance at `t = -1` is reasonably good rather than perfect: standardized mean differences for key inventor characteristics and productivity variables are generally modest, but treated inventors remain somewhat more experienced and productive than controls. That is the right way to present the design: technically serious, transparent about limitations, and credible because it combines careful construction, explicit diagnostics, and cautious interpretation rather than relying on a black-box regression workflow.


## Repository orientation

A clean GitHub version of the project should separate the work into:

- **construction scripts**: build cleaned firm-year and inventor-year / inventor-event panels,
- **analysis scripts**: baseline DiD, event studies, heterogeneity, and advanced estimators,
- **output folders**: regression tables, coefficient plots, and summary CSVs,
- **notebooks**: one main narrative notebook, plus optional shorter diagnostics notebooks.

This notebook is meant to be the **main interpretive notebook** rather than the place where every heavy computation happens.

## Front-facing summary and suggested figures

This notebook is the long-form project document. The concise repository summary lives in `README.md`, but the same headline message is:

- the project builds linked firm-year and inventor-year M&A event-study panels;
- target-side innovation effects are more consistently negative than acquiror-side effects;
- at the inventor level, the strongest acquiror result is **reallocation**—more exploration and short-run movement—rather than a clean productivity gain;
- heterogeneity by inventor position and firm size is economically important.

### Suggested repository figures

**Headline inventor-year x_firm effects**

![Headline inventor-year x_firm effects](../figures/headline_inv_year_xfirm_effects.png)

**Acquiror inventors: exploration after M&A (Sun-Abraham)**

![Acquiror inventors exploration after M&A](../figures/acquiror_exploration_sa_xfirm.png)

**Target inventors: patents after M&A (Sun-Abraham)**

![Target inventors patents after M&A](../figures/target_total_patents_sa_xfirm.png)



## Construction of the dataset: overview and design logic

A major part of the value of this project is the construction itself. The empirical design is only credible if the data pipeline is transparent about how raw patent, accounting, and M&A records are converted into analysis-ready panels.

This section therefore documents the construction in the same spirit as a methods section in an academic paper, but with a stronger emphasis on engineering choices, reproducibility, and economic interpretation. The target reader is a technically literate economist, data scientist, or recruiter who wants to understand not only *what* was estimated, but *how* the underlying research asset was built.

### What the construction is trying to achieve

The pipeline solves four linked problems:

1. **Link inventions to inventors and firms.**  
   Patent records are inventor-centric and assignee-centric, while the firm-level analysis requires a stable public-firm identifier.

2. **Build meaningful innovation measures rather than just patent counts.**  
   The project adds citation-based quality proxies, novelty measures, exploration/exploitation metrics, and technology proximity scores.

3. **Identify economically meaningful inventor moves.**  
   The mobility analysis is not based on one-off assignee changes. It uses a stricter move definition designed to reduce false positives from noisy patent assignment histories.

4. **Create panels aligned to M&A timing.**  
   The end product is not merely a collection of cleaned files. It is a set of firm-year, inventor-year, and inventor-event panels centered on merger timing and suitable for DiD and event-study analysis.

### Why a multi-stage pipeline is necessary

No single source contains the final object needed for the project.

- **PatentsView** provides patent, inventor, assignee, location, CPC, and citation information.
- **External patent-quality data** add KPSS-based measures such as `xi_real` and citation fields.
- **Compustat** provides firm fundamentals.
- **WRDS link tables** connect Compustat firms to CRSP/market identifiers such as `permco`.
- **SDC M&A data** provide acquisition and target events.

The pipeline therefore moves from raw source-specific files to increasingly integrated research objects:

> raw patent and firm data → enriched patent-inventor-firm file → mobility and technology measures → final firm-year and inventor-event analysis panels

### Construction philosophy

Three choices are worth stating up front.

**First, the pipeline is intentionally conservative about identifiers.**  
The core analytical firm identifier is `permco`, because it gives a stable public-firm identifier that can be used across patents, market data, and M&A events.

**Second, intermediate files are cached aggressively.**  
This is partly a speed issue and partly a reproducibility issue. Some steps are expensive enough that a serious empirical workflow benefits from explicit checkpoints.

**Third, the split repository structure preserves the logic of the original monolithic construction script.**  
The runner executes modules in sequence using a shared namespace. That is not the most object-oriented design possible, but it is a sensible choice for a research codebase where preserving validated logic is often more important than abstract elegance.



## Construction pipeline at a glance

The construction code is organized into eight sequential modules plus one runner script.

| File | Main purpose | Main outputs |
|---|---|---|
| `run_construction.py` | Executes all construction modules in order | End-to-end pipeline |
| `01_setup_helpers.py` | Paths, imports, global helpers, data download helper | Shared environment |
| `02_patent_panel_construction.py` | Builds the core patent-inventor-firm dataset and patent-level measures | `pat_inv_firm_df.pkl` and intermediate patent objects |
| `03_exploration_exploitation.py` | Adds exploration/exploitation metrics | enriched patent-inventor-firm file |
| `04_mobility_and_mover_metrics.py` | Detects inventor moves and builds move-related performance objects | mover event and move-performance files |
| `05_technology_similarity.py` | Computes technology proximity and rolling similarity measures | event-based and rolling similarity outputs |
| `06_firm_fundamentals.py` | Builds Compustat-based firm fundamentals and links them to `permco` | linked Compustat panel |
| `07_linking_and_manda.py` | Cleans and links M&A deals and adds pre-deal technology similarity | `manda.pkl` |
| `08_final_panels.py` | Produces the final firm-year, firm-event, inventor-year, and inventor-event panels | final analysis panels |

### Pipeline orchestration

The runner preserves the original notebook-like dependency structure:

```python
EXECUTION_ORDER = [
    "01_setup_helpers.py",
    "02_patent_panel_construction.py",
    "03_exploration_exploitation.py",
    "04_mobility_and_mover_metrics.py",
    "05_technology_similarity.py",
    "06_firm_fundamentals.py",
    "07_linking_and_manda.py",
    "08_final_panels.py",
]
```

and then executes the files in a **shared global namespace**:

```python
shared_globals = runpy.run_path(
    str(script_path),
    init_globals=shared_globals,
    run_name="__main__",
)
```

This is a deliberate research-engineering compromise. It keeps the split files readable and GitHub-friendly while minimizing the risk of introducing logic changes during refactoring.



## Step 0. Environment, paths, and helper infrastructure

The setup module does four jobs:

1. loads the core Python stack,
2. defines the project paths,
3. validates that required local inputs exist,
4. provides a helper to download PatentsView files on demand.

### Main libraries used

This construction is fundamentally a **Python data engineering and empirical research pipeline**. The main tools are:

- **pandas** and **NumPy** for table manipulation and vectorized operations,
- **scikit-learn** for matching support and nearest-neighbor logic,
- **tqdm** for progress monitoring in long-running loops,
- **requests** and **zipfile** for downloading and unpacking PatentsView files,
- **collections.Counter / defaultdict / deque** for efficient rolling technology-history objects.

The setup code also defines a strict path check:

```python
def assert_required_paths_exist():
    required_paths = [
        BASE_PROJECT_PATH,
        RAW_DATA_PATH,
        FINANCIAL_DATA_PATH,
        MANDA_DATA_PATH,
        LINKTABLE_CSV,
    ]
    for path in required_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required path does not exist: {path}")
```

This matters because the project depends on several local raw-data directories. Failing early is better than discovering a missing file after several hours of preprocessing.

### On-demand raw-data loading

A useful design choice is the helper that downloads PatentsView files only if they are not already stored locally:

```python
def download_and_load_patentsview_data(file_name, **kwargs):
    base_url = 'https://s3.amazonaws.com/data.patentsview.org/download'
    local_file_path = os.path.join(RAW_DATA_PATH, file_name)
    if os.path.exists(local_file_path):
        print(f"Loading '{file_name}' from local directory.")
    else:
        r = requests.get(f"{base_url}/{file_name}.zip", timeout=300)
        r.raise_for_status()
        with zipfile.ZipFile(BytesIO(r.content)) as z:
            z.extractall(RAW_DATA_PATH)
    return pd.read_csv(local_file_path, delimiter="\t", ...)
```

Economically, this step is mundane. Methodologically, it is important. It makes the project reproducible from raw public patent files instead of relying on opaque manual extracts.

### Why this matters for the notebook narrative

For an employer-facing research notebook, this section demonstrates that the project is not just a set of regressions. It is a full empirical data asset with explicit input validation, caching, and recoverability.



## Step 1. Build the core patent-inventor-firm dataset

This is the foundational construction step. Everything later in the project depends on producing a clean patent-level file that links:

- a patent,
- its inventor(s),
- the public firm to which it can be assigned,
- the patent's technological and quality characteristics.

### Raw patent inputs

The construction draws the following core tables from PatentsView:

- inventor-patent links,
- assignee-patent links,
- inventor locations,
- CPC technology classifications,
- patent-to-patent citations,
- patent issue dates,
- application filing dates.

The code begins with a cache-first wrapper:

```python
def build_or_load_pat_inv_firm(cache_path=None, rebuild=False):
    if (not rebuild) and os.path.exists(cache_path):
        return pd.read_pickle(cache_path)
```

This is good empirical practice because the patent core is both expensive and stable. Once validated, it should not be rebuilt unless upstream logic changes.

### Cleaning and identifier discipline

Several cleaning choices are substantive rather than cosmetic.

#### 1. Keep business assignees and the primary assignee
The assignee file is filtered to:

```python
assignee_df = assignee_df[
    (assignee_df['assignee_type'] == 2) &
    (assignee_df['assignee_sequence'] == 0)
].copy()
```

This means the construction is intentionally focused on the primary business assignee rather than all possible assignee relationships. That is a defensible choice because the downstream goal is to connect patents to publicly listed firms in a way that is stable enough for event-study analysis.

#### 2. Use filing year rather than issue year for many research objects
The code merges issue dates and application dates and defines:

```python
patent_df['filing_year'] = patent_df['filing_date'].dt.year
```

This is economically sensible because filing dates are closer to the timing of inventive activity than grant dates, which can be delayed by examination and administrative processes.

#### 3. Retain detailed CPC information
The code keeps CPC subclasses and groups, using primary CPC assignments for some tasks and full CPC combinations for novelty construction later. This matters because technology direction is central to the project's mechanism story.

### External enrichments

Two external merges are especially important.

#### KPSS patent-quality data
The construction merges patent-level `xi_real` and citation information from an external replication dataset and links patents to `permco`. This is how the project transitions from patent objects to firm-identified patent objects.

#### State-level noncompete enforceability
The code merges a state-year noncompete enforcement score using inventor state and filing year:

```python
pat_inv_firm_df = pat_inv_firm_df.merge(
    nca_df[['filing_year', 'state_fips', 'std_score']],
    on=['filing_year', 'state_fips'], how='left'
).rename(columns={'std_score': 'nca_enforce_score'})
```

This is a useful example of economically informed enrichment. The project is about inventor mobility, so including a local legal environment relevant to mobility is conceptually well motivated.

### Patent-level quality measures

The construction does not rely on one patent-quality measure. It builds several.

#### Forward citations
First, unconditional forward citations are computed from citation links. Then the code constructs class-year normalized bins based on CPC subclass and filing year:

```python
stats = df.groupby(['filing_year', 'cpc_subclass'])['forward_citations']           .quantile([0.90, 0.99]).unstack()
```

This is good measurement design. Raw citations are noisy across technology classes and cohorts. Ranking patents within technology-year cells makes the quality proxy more comparable.

#### KPSS-based citation bins
The same binning logic is repeated for the KPSS citation field. This redundancy is useful: it reduces dependence on a single citation definition.

### Patent novelty

The novelty measure is based on new combinations of CPC classes, following the logic of recombinatorial innovation. The key idea is simple:

- represent a patent as a set of technology classes,
- enumerate all within-patent class pairs,
- identify whether each pair is new in the historical record,
- summarize the share of new combinations.

A simplified version of the core logic is:

```python
def _canonical_pair(a, b):
    return f'{a}_{b}' if a <= b else f'{b}_{a}'

pairs = g.assign(pair_key=lambda df: df['cpc_tuple'].apply(_make_pairs)).explode('pair_key')
first_date_per_pair = pairs.groupby('pair_key')['issue_date_dt'].min()
```

This is an intellectually strong part of the construction because it goes beyond volume and average quality. It directly measures whether the patent recombines knowledge in a way that appears novel relative to prior art.

### Citation-link measures

The pipeline also builds:

- backward citations,
- self-citations at the firm level.

The self-citation routine is written carefully to be memory-safe by operating in chunks and comparing assignee sets through set intersections. That is a strong engineering choice for large citation graphs.

### Final patent-inventor-firm object

After all merges, the project creates the core research object:

```python
pat_inv_firm_df = pat_inv_df.merge(patent_df, on='patent_id', how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(kpss_df, on=['patent_id'], how='inner')
pat_inv_firm_df = pat_inv_firm_df.merge(patent_novelty_df, on='patent_id', how='left')
...
```

It then adds inventor career features such as:

- first filing year,
- first CPC field,
- inventor age measured as years since first filing,
- indicators for whether a patent stays close to the inventor's original field.

### Why this step is done this way

This construction step is designed to create a **single enriched micro-level innovation file** from which multiple downstream panels can be derived without rebuilding patent logic each time. That is both computationally efficient and methodologically cleaner.



## Step 2. Construct exploration and exploitation measures

One of the most interesting aspects of the project is that it tries to distinguish **direction of search** from **quantity of output**.

The key intuition is that mergers may reshape what inventors and firms work on even when the total number of patents does not move much.

### Measure design

For each patent, the code builds a set of CPC subclasses and compares that set to the recent knowledge base of:

- the patent's inventor team,
- the patent's firm or firms.

The knowledge base is defined using a rolling five-year window.

A simplified version of the logic is:

```python
for row in patent_level.itertuples(index=False):
    inv_knowledge = union_of_recent_cpcs_for_inventors
    firm_knowledge = union_of_recent_cpcs_for_firms

    exp_inv = 1 - overlap(current_cpcs, inv_knowledge) / len(current_cpcs)
    exp_firm = 1 - overlap(current_cpcs, firm_knowledge) / len(current_cpcs)
```

### Why this is economically meaningful

A patent is classified as more exploratory when it uses CPC classes that are less represented in the inventor's or firm's recent history. This is an intuitive way to measure movement into less familiar technological space.

This is also one reason the project's headline story is more persuasive for exploration than for aggregate patent counts. Exploration is closer to the organizational mechanism one might expect after an acquisition:

- new teams are combined,
- internal allocation changes,
- inventors may be redeployed,
- firms may search more broadly or in adjacent spaces.

### Why the implementation uses rolling deques

The implementation stores prior technology histories in `deque` objects and purges outdated years as it moves through the patent sequence. That makes the rolling-window computation efficient enough to scale while remaining transparent.



## Step 3. Identify inventor mobility events and build mover metrics

A central contribution of the project is that inventor mobility is not treated as a noisy byproduct. It is directly measured and then connected to post-M&A firm outcomes.

### Strict move identification

The move definition is intentionally conservative. The code first restricts attention to inventors with at least five patents:

```python
min_patents = 5
prolific_inventors = inventor_counts[inventor_counts >= min_patents].index
```

Then it defines a move only when there is a stable firm sequence around the transition:

- two patents before the move at the old firm,
- the first patent at the new firm,
- two subsequent patents at the new firm.

The core rule is:

```python
is_move = (
    (career_df['permco'] != career_df['prev_permco']) &
    (career_df['prev2_permco'] == career_df['prev_permco']) &
    (career_df['next_permco'] == career_df['permco']) &
    (career_df['next2_permco'] == career_df['next_permco'])
)
```

### Why this strict rule is useful

Patent assignment data can be noisy. A single assignee switch may reflect legal assignment timing, joint work, or temporary data noise rather than a real labor-market transition.

The stricter move rule sacrifices some sample size, but it likely improves signal quality, which is exactly the right tradeoff for a project whose main mechanism involves labor reallocation.

### Pre/post mover performance

Once moves are defined, the code builds five-year pre-move and post-move performance windows and aggregates inventor outcomes such as:

- patent counts,
- citations,
- `xi_real`,
- novelty,
- backward and self-citations,
- exploration and exploitation,
- team size.

This produces both:
- a long panel with one row per inventor-move-period, and
- a wide format that makes change calculations straightforward.

### Benchmarking movers relative to peers

The project also compares movers to peers at origin and destination firms. That is a subtle but valuable addition because it helps distinguish:

- whether firms are losing unusually important inventors,
- whether incoming inventors look stronger or weaker than incumbent peers,
- whether post-move performance changes suggest integration frictions or gains.

From a research-design perspective, this is a strong bridge between micro labor allocation and firm-level organizational outcomes.



## Step 4. Measure technology similarity around moves and deals

The project next constructs technology-similarity objects that help interpret whether mobility and M&A connect technologically related or unrelated domains.

### Event-based proximity around inventor moves

The first routine compares technology vectors before and after a move for:

- the inventor,
- the origin firm,
- the destination firm.

Vectors are represented as `Counter` objects over CPC subclasses, and similarity is measured with cosine similarity:

```python
def _calculate_cosine_similarity(vec1, vec2):
    dot_product = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in all_keys)
    norm1 = sqrt(sum(v**2 for v in vec1.values()))
    norm2 = sqrt(sum(v**2 for v in vec2.values()))
    return dot_product / (norm1 * norm2)
```

This yields quantities such as:

- inventor pre/post self-similarity,
- inventor similarity to origin firm,
- inventor similarity to destination firm,
- origin-destination firm similarity.

### Rolling annual similarity

The second routine creates a rolling annual similarity measure, comparing a current year's technology vector with the preceding five-year technology history. This is helpful because it turns technology direction into a panel variable rather than a one-time event summary.

### Why these measures matter

These objects are not the headline outcomes of the project, but they greatly improve interpretation. If mobility rises after deals, the next question is whether it reflects:

- redeployment into familiar technologies,
- integration into the acquiring firm's technological base,
- broader exploratory search.

Technology similarity measures help answer that question.



## Step 5. Build firm fundamentals from Compustat and link them to market identifiers

The firm-side empirical analysis needs more than patent outcomes. It needs standard controls and heterogeneity variables describing firm size, profitability, leverage, liquidity, investment, and financial constraints.

### Sample construction in Compustat

The code filters Compustat to a standard industrial sample:

- excludes financials, utilities, and public sector entities,
- keeps industrial, standardized, consolidated, domestic statements,
- removes clearly invalid negative values for core accounting variables.

It then defines the analysis year as:

- same calendar year if the fiscal year ends in June to December,
- previous calendar year otherwise.

That timing rule is sensible because it aligns fiscal records more closely with the year to which they economically belong.

### Feature engineering

The module then constructs a broad set of variables, including:

- size: `log_at`, `log_sale`, `log_mv`,
- profitability: `roa`, margins, operating profitability,
- growth and valuation: `sale_growth`, `tobinsq`, `market_to_book`,
- financing and liquidity: leverage, cash, interest coverage,
- investment and composition: capital expenditures, R&D intensity, intangible assets,
- payout and repurchases,
- constraint indices such as **Whited-Wu** and **Hadlock-Pierce**.

This is more than a cosmetic merge. It creates the economic state variables needed to:
- control for confounding firm conditions,
- define heterogeneity splits,
- interpret M&A effects in a richer organizational context.

### Real scaling and CPI adjustment

The code downloads CPI data from FRED and uses it to build real assets and real R&D variables. That is another sign that the construction is meant to support serious empirical work rather than only descriptive charts.

### Linking Compustat to `permco`

The link to the patent side is done using the CRSP-Compustat link table. The code keeps primary, validated links and restricts them to valid date ranges.

```python
compustat_core_linked = compustat_core_df.merge(
    linktable[['gvkey', 'permco', 'linkdt', 'linkenddt']],
    on='gvkey', how='inner'
)
compustat_core_linked = compustat_core_linked[
    (compustat_core_linked['datadate_dt'] >= compustat_core_linked['linkdt']) &
    (compustat_core_linked['datadate_dt'] <= compustat_core_linked['linkenddt'])
]
```

This link discipline is essential. The project would be much weaker if the public-firm mapping were fuzzy or time-inconsistent.



## Step 6. Clean and link M&A deals

The M&A module transforms raw SDC transaction records into a deal panel that can be merged into firm-year and inventor-year data.

### Deal cleaning choices

Several choices are worth highlighting.

#### 1. Announcement timing is central
The project uses announcement-year timing for the event-study setup. That is reasonable because announcement is typically the point at which firms, investors, and employees begin responding to the transaction.

#### 2. Link both target and acquiror through CUSIP-6 and valid date windows
The code truncates CUSIPs to six digits and merges both sides of the deal through the link table. It then keeps only observations for which the announcement date falls within the valid link ranges for both firms.

#### 3. Restrict deal outcomes
The code keeps completed and withdrawn deals and explicitly creates a failed-merger indicator. This is useful because failed deals can later be used either as controls, robustness checks, or descriptive contrasts.

### Deal-level variables

The M&A panel retains:

- acquiror and target identifiers,
- transaction value,
- ownership percentages,
- diversifying indicators based on industry codes,
- deal outcome and failure status.

### Pre-deal technology similarity between target and acquiror

A particularly strong addition is the pre-deal technology similarity measure, built from five-year patent portfolios of the target and acquiror.

This is useful for at least two reasons:

1. it helps characterize whether deals combine related or more distant technology portfolios,
2. it provides a natural heterogeneity variable for later interpretation.

In other words, the M&A panel is not just a timing file. It is a rich deal-characterization file.



## Step 7. Assemble the final analysis panels

The final module turns the intermediate construction objects into the panels used for estimation.

This is the point where the project shifts from **data integration** to **econometric design**.

### 7.1 Firm-year panel

The code first aggregates patent outcomes to the `permco × year` level:

```python
firm_year_patent_metrics = pat_inv_firm_df.groupby(['permco', 'filing_year']).agg(
    total_patents=('patent_id', 'nunique'),
    cites=('cites', 'sum'),
    xi_real=('xi_real', 'sum'),
    ...
).reset_index()
```

It then merges in:

- rolling firm technology similarity,
- Compustat fundamentals,
- inventor arrival and departure measures,
- relative quality of arriving and departing inventors,
- M&A event counts and deal-value measures.

The result is a firm-year panel that simultaneously describes:
- innovative output,
- technology direction,
- labor flows,
- financial conditions,
- M&A exposure.

That integration is one of the most compelling aspects of the project.

### 7.2 Firm-year M&A event-study panel

The next object is a firm-year event-study panel centered on the **closest deal year** within a ±5-year window.

A nice engineering feature here is that the code avoids `merge_asof` and instead computes previous and next deal indices explicitly with `np.searchsorted`. That makes the logic more transparent and less fragile.

The panel then defines:

- `ma_deal_role` = acquiror, target, or no recent M&A,
- `years_from_ma_deal`,
- deal value,
- failed-merger flag,
- other-party identifier,
- pre-deal technology similarity.

This object is the main firm-level treatment panel used for DiD and event-study work.

### 7.3 Inventor-year panel

The inventor-year panel is built for inventors who ever patent at firms in the analysis universe. The code creates:

- annual inventor outcome measures,
- zero-filled years for no-patenting observations within the inventor career span,
- annual employer assignment,
- move-year indicators,
- M&A context of the assigned employer.

This is an ambitious and important design choice. It converts sparse patent observations into a panel suitable for longitudinal analysis of inventor behavior.

### 7.4 Inventor-event panel with matched controls

The most sophisticated construction step is the inventor M&A event-study panel.

The logic is:

1. identify treated inventor-deal events,
2. keep only events feasible over the full event window,
3. construct matched control pseudo-events using inventor characteristics at `t = -1`,
4. expand each event into a balanced `[-5, +5]` relative-year grid,
5. merge annual inventor outcomes back onto the event grid.

The matching features include variables such as:

- inventor age,
- cumulative patents,
- cumulative citations,
- first CPC subclass.

The code supports propensity-score-based matching with exact matching on selected categorical fields.

This is a strong design because it tries to build a credible inventor-level counterfactual while keeping the event-study object balanced and interpretable.

### Why these final panels are the right endpoint

By the end of the pipeline, the project has produced four conceptually distinct but connected research objects:

1. **firm-year panel** for aggregate organizational outcomes,
2. **firm event-study panel** for treatment-timing analysis,
3. **inventor-year panel** for individual-level longitudinal behavior,
4. **inventor-event panel** for matched event-study analysis.

That architecture reflects a PhD-level empirical strategy: it links micro mechanism evidence to firm-level outcomes rather than relying on only one level of aggregation.


## Construction diagnostics: what the final balancing tests imply

The completed construction run also produces useful diagnostics for the inventor M&A event-study panel. These are worth discussing explicitly because they speak not only to code correctness, but to the credibility of the matched design that underlies the inventor-side analysis.

### Event-time balance and panel integrity

The inventor event-study panel contains **8,243,257 rows and 48 columns**, with **441,242 balanced inventor-events out of 441,242 total inventor-events** over the full `[-5, +5]` window. This is an important confirmation that the event-panel construction worked as intended.

In practice, that means every retained inventor-event contributes the full eleven relative years required for the baseline event-study design. The reported row counts by relative time are perfectly stable across the window:

- treated rows per event time: **210,521**
- control rows per event time: **538,866**

That stability is exactly what one hopes to see in a stacked matched event-study panel. It suggests that no relative years are being dropped asymmetrically after matching, reshaping, or outcome merging, and that the final inventor-event object is structurally well behaved for downstream estimation.

### What the balance checks say

At `t = -1`, the matched treated and control groups are reasonably close on the main inventor characteristics used for matching and interpretation:

- `inventor_age`: treated = **10.531**, control = **9.952**, SMD = **0.098**
- `cum_patents`: treated = **9.776**, control = **7.631**, SMD = **0.144**
- `cum_cites`: treated = **346.441**, control = **197.801**, SMD = **0.129**
- `total_patents`: treated = **0.970**, control = **0.728**, SMD = **0.105**
- `cites`: treated = **25.157**, control = **11.824**, SMD = **0.071**

The overall picture is encouraging rather than perfect. Standardized mean differences around `0.07` to `0.14` indicate that the matching substantially improves comparability, but does not eliminate all pre-existing differences between treated inventors and controls. In particular, treated inventors remain somewhat more established and somewhat more productive than controls even before the event. That is not surprising in this setting—high-value inventors are plausibly more exposed to acquisition and target-side reorganization—but it is something the notebook should state openly.

The right interpretation is therefore that the matched design creates a much more credible counterfactual than an unmatched comparison, while still leaving room for residual selection on inventor quality or career trajectory. That is precisely why the project benefits from event-study pre-trend tests and from advanced staggered-treatment estimators such as Sun–Abraham and Borusyak–Jaravel–Spiess.

### Control reuse: acceptable, but worth being explicit about

The diagnostics also show substantial reuse of matched controls:

- unique control-event matches: **487,373**
- share of controls reused more than once: **0.779**
- maximum reuse count: **132**

This is not a construction failure. It is a common consequence of nearest-neighbor matching in a large treated-event sample, especially when the design tries to keep controls comparable on age, cumulative productivity, and technological background. Some control inventors are simply very attractive matches for many treated events.

Still, the reuse should be described candidly. The raw row count in the final panel is very large, but effective independent variation is somewhat smaller than that row count might suggest because certain controls appear repeatedly. In practical terms, this does not invalidate the design, but it does strengthen the case for conservative inference and for reading the matched sample as a tool for improving comparability rather than a guarantee of perfect one-to-one counterfactual assignment.

### Bottom line on the construction diagnostics

Taken together, the final diagnostics support the view that the inventor event-study panel is **successfully constructed, balanced in event time, and reasonably well matched on observables**, while still reflecting the familiar empirical reality that treated inventors are somewhat positively selected and that donor controls are reused heavily.

That is a good outcome for a research repository. It means the project can present the inventor-side design as a serious and credible quasi-experimental object without overstating what matching alone can accomplish. In a public-facing notebook, I would frame these diagnostics as a strength: the project does not merely run estimators, but documents the quality and limits of the underlying design in a transparent way.



## Construction choices that are especially strong, and a few places to be explicit about limitations

### What is especially strong in the construction

1. **The project builds real research infrastructure, not just a regression-ready CSV.**  
   The layered construction from raw sources to multiple final panels is one of the clearest signs of technical maturity in the repository.

2. **The measurement choices are economically motivated.**  
   Novelty, exploration, mobility, and technology similarity are not generic machine-learning features. They are tailored to the economic mechanisms of the question.

3. **The pipeline balances ambition with tractability.**  
   Intermediate caching, chunked citation routines, and explicit path validation show a practical understanding of what large empirical projects require.

4. **The final objects are designed for multiple estimators.**  
   The construction is clearly shaped by downstream DiD, event-study, and matching requirements.

### A few limitations or caveats worth stating openly

A strong notebook should also be explicit about where the construction is deliberately imperfect.

- **Public-firm scope.**  
  The use of `permco` is analytically powerful, but it means the project is strongest for the publicly listed-firm universe rather than the full economy of patent assignees.

- **Assignee-based employer inference.**  
  Employer assignment from patent assignees is sensible and standard in this setting, but it is still an inferred labor-market link rather than a direct HR record.

- **Strict move definitions trade off recall for precision.**  
  That is a feature, not a bug, but it should be stated clearly.

- **The split modules still rely on sequential shared-state execution.**  
  For a research repository, this is acceptable and often desirable for faithfulness. For a production software package, one would likely move further toward explicit function imports and typed interfaces.

### One issue I would state honestly

The setup file still contains a comment suggesting that some foundational economic-data loading is "assumed to be here for brevity." In the current split repository, those later steps are handled by subsequent modules. This does not appear to break the pipeline, but the notebook should describe the actual implemented sequence rather than the older monolithic-script commentary.

That is an example of the kind of small inconsistency worth acknowledging rather than hiding.



## Summary of the construction

The construction pipeline creates a research dataset that is substantially richer than a standard event-study input file.

### In one sentence

The project starts from raw patent, accounting, and deal records and builds a linked set of firm-level and inventor-level panels that can speak to both **organizational outcomes** and **microeconomic mechanisms** around M&A.

### What each stage contributes

- **Setup and orchestration** establish a reproducible research environment.
- **Patent panel construction** creates the core patent-inventor-firm microdata.
- **Quality and novelty measures** turn patents into economically meaningful innovation objects.
- **Exploration and exploitation metrics** capture the direction of technological search.
- **Mobility construction** identifies meaningful inventor moves and summarizes mover quality.
- **Technology similarity measures** help interpret whether movement reflects related or distant technological reallocation.
- **Firm fundamentals and link tables** anchor the patent side in standard corporate-finance measurement.
- **M&A linking** creates timed treatment events with deal characteristics.
- **Final panels** turn all of the above into objects suitable for DiD, event studies, heterogeneity analysis, and matched inventor-event designs.

### Why this matters for the broader project

For recruiters and economists at technology firms, the construction is important because it demonstrates a combination of skills that often appear separately but less often together:

- research design,
- large-scale data construction,
- identifier linking across sources,
- thoughtful feature engineering,
- sensitivity to measurement error,
- and an ability to structure the final output around a clear economic question.

That combination is a major part of what makes the project a credible showcase of independent empirical research skill.


## Analysis workflow: overview and design logic

The analysis side of the project is organized much more explicitly than the construction side. The repository-facing analysis code does **not** recreate the original monolithic script by sharing a live namespace across modules. Instead, it uses a cleaner split between configuration, data loading, reusable estimator functions, and two high-level runners:

- a **firm-panel branch**, and
- an **inventor-year / inventor-event branch**.

That choice is economically and computationally sensible. The firm-level and inventor-level designs answer related but distinct questions, use different units of observation, and have different memory bottlenecks. Keeping them separate makes the workflow easier to inspect and easier to explain to external readers.

At a high level, the logic is:

1. load the final panels produced by construction,
2. define the treatment and event-time structure,
3. run baseline two-way fixed-effects DiD and event studies,
4. layer on heterogeneity specifications,
5. run selected staggered-adoption robust estimators,
6. use placebo and balance diagnostics to decide what is credible,
7. interpret only the results that survive those checks.

That last step is important. The notebook should not present the analysis as “run many methods until something is significant.” It should present the analysis as **triangulation**: multiple estimators are used because the treatment is staggered, treatment effects can be heterogeneous, and simple TWFE event studies can be misleading.


## Analysis pipeline at a glance

The public workflow is intentionally split into small files that do one thing each.

### Core orchestration
- `run_analysis.py` calls the two main branches in sequence.
- `run_firm_panel_analysis.py` contains the full firm-level workflow.
- `run_inventor_year_analysis.py` contains the inventor-year workflow.

### Shared infrastructure
- `config.py` defines paths, windows, and global switches.
- `data.py` loads the cleaned panels and merges lagged firm controls into the inventor-level datasets.
- `utils.py` contains the reusable econometric helpers such as the two-way fixed-effects `PanelOLS` wrapper and the heterogeneity `Z` constructors.

### Topical method modules
- `firm_analysis.py` contains the baseline firm DiD/event-study workflow, the matched stacked design, and the generic triple-DiD/event-study machinery.
- `inventor_year.py` contains inventor-event preparation, within-firm heterogeneity splits, downsampling for advanced methods, and the inventor-year CSDID wrapper.
- `advanced_methods.py` contains the advanced estimators used as cross-checks: Causal Forest, Synthetic Control, Sun–Abraham, and Borusyak–Jaravel–Spiess.
- `placebos.py` contains lead and permuted placebo timing exercises.
- `summary_statistics.py` writes compact panel summaries and pre-period baselines.

This split is a strength of the repository version. It makes the analysis look like a serious research pipeline rather than a notebook that happened to work once.


## Step A. Configuration and loading of the analysis data

The analysis starts with a compact configuration object:

```python
@dataclass
class AnalysisConfig:
    analysis_window: tuple[int, int] = (1980, 2020)
    event_study_window: tuple[int, int] = (-5, 5)
    event_study_ref_k: int = -1
    run_inventor_year_advanced: bool = True
    advanced_alpha: float = 0.05
    inventor_year_csdid_bootstrap_draws: int = 30
```

This is good practice for a public research project. It puts the key design choices in one place:

- the **calendar window** for the usable panel,
- the **event-study window**,
- the omitted event-time reference period,
- and the switches controlling the more expensive advanced methods.

The data loader then reads the final construction outputs and attaches lagged firm controls to the inventor-side panels. That is a subtle but important design step. The inventor outcomes are naturally inventor-level, but the mechanism may still run through the characteristics of the employing firm. By merging lagged firm controls into the inventor panels, the project can ask whether inventor-level changes persist after accounting for the employer's pre-period scale and financial condition.

Conceptually, this means the analysis combines:
- **individual outcomes**,
- **firm event timing**, and
- **firm baseline controls**.

That is exactly the kind of multi-level empirical setup that signals strong applied micro / innovation-economics training.


## Step B. Firm-panel design: what the firm analysis is trying to estimate

The firm branch asks a relatively direct question:

> When a publicly listed firm becomes an acquiror or a target, how do its innovation and inventor-flow outcomes evolve relative to otherwise similar firms without a recent M&A event?

The raw event panel is first turned into a standard DiD/event-study input:

```python
did_df["Treated"] = (did_df["ma_deal_role"] != "no_recent_MandA").astype(int)
did_df["Post"] = (did_df["years_from_ma_deal"].fillna(-1) >= 0).astype(int)
did_df["Post_Treated"] = did_df["Treated"] * did_df["Post"]
did_df["gname"] = np.where(
    did_df["years_from_ma_deal"].isna(),
    0,
    did_df["data_year"] - did_df["years_from_ma_deal"],
).astype(int)
```

Economically, these objects do the following:

- `Treated` identifies firms exposed to an M&A event.
- `Post` marks the post-event period.
- `Post_Treated` is the standard average post-treatment effect regressor.
- `gname` stores the cohort year, which is essential for staggered-treatment methods.

The code then creates separate datasets for:
- **acquiror vs. no recent M&A**, and
- **target vs. no recent M&A**.

That separation is a very good design choice. Acquirors and targets should not be pooled mechanically. The organizational mechanisms are different:
- acquirors face integration, reallocation, and coordination decisions;
- targets face absorption, disruption, or reorganization from the receiving side.

Treating them as separate empirical objects gives the analysis a cleaner interpretation.


## Step C. Why the firm analysis uses a matched stacked design

The baseline firm analysis does **not** just run one pooled two-way fixed-effects regression on all firms. It first builds a matched stacked panel.

The matching function works cohort by cohort. At the year before treatment, it uses:
- industry (`sic3`),
- firm size (`log_sale`, `log_mv`),
- and a cohort-specific propensity score,

to pair treated firms with comparable controls. Within industry, the actual pair selection uses Mahalanobis distance on firm size variables, with an optional propensity-score caliper.

A compact summary of the matching logic is:

```python
for cohort_year in cohort_years:
    cov = df[df["data_year"] == (cohort_year - 1)].copy()
    pot = cov[(cov["gname"] == 0) | (cov["gname"] == cohort_year)].copy()

    # estimate cohort-specific propensity score
    lr.fit(X_ps, y_ps)
    pot["pscore"] = lr.predict_proba(X_ps)[:, 1]

    # within sic3, match on Mahalanobis distance in (log_sale, log_mv)
    dist = cdist(X_tr, X_ct, metric="mahalanobis", VI=VI)
```

This is attractive for three reasons.

### 1. It improves comparability
The design does not rely only on fixed effects to clean up large cross-sectional differences. It first tries to compare treated firms to firms that already looked similar before the event.

### 2. It respects treatment timing
Matching is done relative to each treatment cohort rather than once for the whole sample.

### 3. It makes the event-study interpretation sharper
Because the stacked panel is built around comparable treated-control sets for each cohort, the event-study plots are easier to interpret than a single fully pooled TWFE event study.

This is not a magic solution. Matching does **not** prove parallel trends. But it is a strong design enhancement, and for a recruiter or economist reading the notebook it signals that the empirical design is not naïve.


## Step D. Baseline estimation: fixed effects DiD and event studies

The core regression wrapper is intentionally simple and reusable. It runs a `PanelOLS` with:
- entity fixed effects,
- time fixed effects,
- clustered standard errors.

```python
mod = PanelOLS(
    y,
    X,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
    check_rank=True,
)
```

For each outcome, the analysis does two related things:

### 1. Baseline DiD
Estimate the average post-treatment effect through `Post_Treated`.

### 2. Event study
Replace the single post indicator with event-time dummies interacted with treatment status, omitting `k = -1` as the reference year.

That baseline event-study machinery is important because it lets the notebook ask:
- whether effects build slowly or quickly,
- whether pre-trends are approximately flat,
- whether the treatment effect is concentrated in the deal year or after it.

The code also saves compact regression tables and coefficient plots for every outcome. That is good repo design because it makes the project reproducible and inspectable without forcing users to rerun everything.

### Caveat
The notebook should be explicit that plain two-way fixed-effects event studies can be biased under staggered timing and heterogeneous treatment effects. The project handles that appropriately by treating the baseline TWFE results as a **first screen**, not as the final word.


## Step E. Heterogeneity analysis: why the project uses `Z` interactions and triple-DiD-style specifications

A major strength of the project is that it moves beyond average effects. The analysis constructs several pre-treatment heterogeneity variables:

- baseline firm size (`Z_log_sales_cont`, plus quantile bins),
- deal size relative to market value (`Z_deal_rel_cont`, plus quantile bins),
- inventor rank within firm on cumulative patents or citations.

These are generated at the baseline period, typically `k = -1`, and then broadcast over the event window. That is the right way to do it. It keeps the heterogeneity split **predetermined** rather than allowing post-treatment variables to contaminate interpretation.

The generic event-study runner then supports a triple-DiD-style extension:

```python
df_use[post_z] = df_use["Post"] * z_num
df_use[triple] = df_use["Post_Treated"] * z_num
```

In plain language, this asks not only:

> Did treated firms change after the event?

but also:

> Did treated firms change **more or less** after the event depending on baseline size, deal scale, or inventor rank?

This is exactly the kind of heterogeneity design that makes the project look more mature. Tech economists and hiring managers are often less impressed by one average treatment effect than by a disciplined answer to **where** the effect is strongest and **why**.


## Step F. Advanced firm estimators: what each one contributes

The project then re-estimates selected outcomes with more specialized methods. With the current uploaded firm results in hand, these methods are no longer just “nice robustness checks.” They materially change how the firm-level evidence should be interpreted.

### 1. Sun-Abraham interaction-weighted event study
This is the most natural staggered-adoption robustness check for the baseline event studies. It avoids some of the weighting problems of TWFE event studies when treatment timing is staggered and treatment effects vary by cohort or over event time.

Why it matters in this notebook:
- if a pattern is visible in both the matched TWFE event study and Sun-Abraham, confidence increases;
- if Sun-Abraham weakens or flattens a TWFE pattern, the TWFE result should be treated cautiously;
- if large positive leads remain even under Sun-Abraham, the right conclusion is not “causal proof,” but “the underlying timing pattern is complicated.”

That is exactly what happens for some of the target-side patenting and citation outcomes. The baseline TWFE specifications often show economically large post-merger declines, but the advanced event-study results do not line up cleanly enough to justify a one-line “M&A reduces target innovation” claim.

### 2. Borusyak-Jaravel-Spiess imputation estimator
This method imputes untreated outcomes and then constructs treatment effects from those imputed counterfactuals. It is a useful complement to Sun-Abraham because it relies on a different implementation logic.

Why it matters here:
- agreement between BJS and Sun-Abraham is especially reassuring;
- disagreement is informative and should be discussed, not hidden.

In the current firm outputs, disagreement is part of the result. For example, target patent outcomes that look clearly negative in the baseline DiD often become much less negative or even positive in BJS. That does not mean BJS is “right” and everything else is “wrong.” It means average patent effects are estimator-sensitive and should be presented as such.

### 3. Synthetic Control
The code runs firm-specific synthetic control fits and then aggregates relative gaps across successful treated units.

Why it matters:
- SCM is visually intuitive,
- it is useful for showing that some effects are not artifacts of a single regression functional form,
- and it gives a concrete unit-level interpretation.

Main caveat:
SCM is more fragile than it looks. Results depend on donor availability, pre-period fit, window choice, and whether a treated firm has enough support for a sensible synthetic counterpart.

In the current results, SCM is best treated as directional support rather than a headline estimator. It often reinforces that acquiror-side patent effects are weak and that target-side output responses are mixed rather than uniformly negative.

### 4. Causal Forest
The firm analysis also runs a Causal Forest using pre-period covariates and short-run post outcomes.

This is **not** the main identification engine. It is best understood as a structured heterogeneity tool:
- which pre-period firm characteristics predict more negative or more positive treatment effects?
- does the forest agree with the simpler size-based heterogeneity splits?

That is the right way to present it. In the uploaded firm results, the forest ATEs are typically imprecise and centered near zero, which means the forest is **not** strong evidence for an average treatment effect. What it does add is a consistent ranking of moderators: baseline size, liquidity, profitability, and valuation variables repeatedly matter, with firm size emerging as the single most important heterogeneity dimension.


## Step G. Placebo and balance checks: why they are central rather than decorative

A strong empirical notebook should make clear that credibility is built through failed tests as well as successful ones.

The analysis includes:

### Pre/post balance diagnostics
The matching design is evaluated by comparing treated and control firms on pre-period variables such as `log_sale` and `log_mv`, before and after the stacking procedure.

Those diagnostics are useful, but they should not be oversold. In the current upload, the target matches look materially closer than the acquiror matches on simple levels, while the acquiror design still begins from noticeably larger treated firms. That does not invalidate the specification, but it does mean the notebook should lean on within-firm dynamics and robustness rather than on any intuition from raw level comparisons.

### Lead placebos
Treatment is artificially shifted earlier in time. If the model then finds a large effect before the actual event, that is evidence against a clean causal interpretation.

### Permutation placebos
Treatment timing is permuted across treated units. This checks whether the apparent signal is stronger than what one would expect from arbitrary timing assignments.

This is a very good robustness philosophy. It tells the reader:

- we are not just looking for significance,
- we are asking whether the effect looks specific to the actual event timing.

And in the uploaded firm outputs, the placebo results are genuinely informative. For firm-level patent outcomes, the baseline TWFE DiD can generate statistically significant placebo effects under fake timing, especially for acquirors and in one of the target placebo specifications. That materially lowers confidence in any interpretation that relies only on a single significant baseline coefficient.

This is therefore one of the most important messages of the notebook. The project is strongest when it treats placebos as decision tools: if an average effect does not survive placebo skepticism and cross-estimator comparison, it should not be headlined.


## Step H. Inventor-year design: the mechanism side of the project

The inventor-year branch is where the project becomes especially distinctive. It asks a different question from the firm panel:

> What happens to inventors attached to treated firms, relative to matched control inventors, around merger events?

This is the more mechanism-oriented design. The firm panel tells us whether innovation or inventor flows changed at the organization level. The inventor panel tells us whether those firm-level changes are associated with:
- inventor mobility,
- changes in exploration versus exploitation,
- and changes in individual inventive output.

The workflow cycles over:
- role = `acquiror` or `target`,
- whether lagged firm controls are included,
- whether inventor-side controls are included,
- and several heterogeneity layers.

That branching structure is not accidental. It is a way of checking whether the substantive conclusions survive moderate changes in conditioning information.


## Step I. Preparing the inventor event panel: treated inventors versus matched controls

The inventor panel is built from the balanced inventor-event-study panel created in construction. The analysis then extracts a role-specific sample such as “acquiror inventors vs. control anchors” or “target inventors vs. control anchors.”

A central preparation step is:

```python
df = df[(df["ma_deal_role"] == role_l) | (df["is_control_event"] == 1)].copy()
df["Treated"] = (df["ma_deal_role"] == role_l).astype(int)
df["event_time"] = pd.to_numeric(df["years_from_ma_deal"], errors="coerce").astype(float)
df["cohort"] = pd.to_numeric(df["closest_deal_year"], errors="coerce").astype(float)
df["Post"] = (df["event_time"] >= 0).astype(int)
df["Post_Treated"] = df["Post"] * df["Treated"]
```

The identification logic is therefore parallel to the firm panel, but the unit of analysis is now an inventor-event rather than a firm-year.

A particularly strong feature is that the code can also create **within-firm rank indicators** based on cumulative patents or citations at `t = -1`. That makes it possible to ask whether higher-ranked inventors respond differently to M&A than lower-ranked inventors within the same firm context.

That is a very appealing design feature for a job-market or recruiter-facing notebook because it shows attention to the internal composition of firms, not just the average inventor.


## Step J. Baseline inventor outcomes and why they matter

The inventor-year branch focuses on a deliberately interpretable set of outcomes:

- `total_patents`
- `cites`
- `xi_real`
- `novelty_score_group`
- `exploration_inv`
- `exploitation_inv`
- `is_move_year`

This is a good outcome list. It spans three conceptually different margins:

### 1. Mobility
`is_move_year` is a direct mechanism outcome. If M&A affects organization, incentives, or match quality, inventor moves are one of the most natural places to look.

### 2. Direction of search
`exploration_inv` and `exploitation_inv` speak to whether inventors stay in familiar technology areas or move into newer ones.

### 3. Individual output and influence
Patents, citations, and `xi_real` capture different notions of inventor productivity and impact.

This mix is more persuasive than relying only on patent counts. It shows the reader that the project is not treating innovation as a one-dimensional object.


## Step K. Inventor-year advanced methods and why the project uses them selectively

The inventor panel is much larger and more computationally demanding than the firm panel, so the advanced methods are used more selectively.

### 1. CSDID / Callaway–Sant'Anna
The code runs a compact wrapper around `ATTgt`, using role-specific cohort definitions such as:
- `cs_g_year_target`,
- `cs_g_year_acquiror`,
- `cs_g_year_all`.

This is a strong choice. Inventor exposure to treatment is genuinely staggered, and a cohort-based estimator is appropriate. The code also trims weakly identified cells and keeps the post horizon modest, which is exactly what one should do when the panel is large and some cohorts are thin.

### 2. Sun–Abraham
For outcomes that are significant in the baseline inventor specification, the code reruns Sun–Abraham as a robustness check on the inventor-event panel.

### 3. Optional BJS
The inventor workflow is set up so that BJS can also be run, but the public configuration keeps this limited because of runtime and memory considerations.

### 4. Downsampling for advanced methods
The project explicitly downsamples inventor-event units before running the heavier estimators.

That is a sensible engineering compromise, not a conceptual weakness, as long as the notebook is honest about it. The right way to present it is:

- baseline DiD/event-study uses the full prepared panel;
- advanced methods are used on a carefully downsampled but balanced subset to keep the workflow feasible.

That explanation reads as practical and credible.


## Step L. What the current results appear to say

This section can now be stated more directly because the uploaded outputs cover the main firm-level baseline, event-study, advanced-estimator, and placebo files. The key conclusion is **not** that every result points in the same direction. The key conclusion is that the project contains a clear difference between results that are robust enough to headline and results that are informative but estimator-sensitive.

### 1. The firm panel does not support a simple universal-output story
On first pass, the firm-level baseline DiD suggests meaningful post-merger declines for targets in log patents, log citations, and log exploration, while acquiror-side effects are weaker. That initial pattern is economically plausible: target firms are more likely to experience disruption, integration, and loss of autonomy.

But the notebook should stop there and ask the harder question: how much of that pattern survives when the estimator changes and when fake timing is tested?

For patents and citations, the answer is: **not enough to justify a sweeping average-effect claim.** The target-side TWFE baseline is economically strong, yet Sun-Abraham still shows large positive leads, BJS often weakens or reverses the post effect, and the placebo exercises show that significance can arise under artificial timing. Acquiror-side patent effects look even weaker and are hard to defend as a clean average treatment effect.

### 2. The firm panel is strongest on reorganization and heterogeneity, not on a single patent collapse coefficient
The cleaner firm-level message is that mergers appear to reorganize innovative activity, and that the magnitude and direction of disruption depend on firm characteristics.

Three pieces of evidence support that interpretation.

First, firm-level exploration responds more coherently than aggregate patent counts. For acquirors, log exploration becomes mildly negative in the post period with relatively flat pre-trends. For targets, exploration also weakens in the baseline specifications, although the target-side timing pattern is noisier than one would want for a definitive causal claim.

Second, inventor-flow outcomes and inventor-count outcomes do not line up with a simple “everything contracts” story. Some advanced-method outputs suggest that staffing and mobility responses may be positive or mixed even when baseline patent output looks weaker. That is exactly what one would expect in a reallocation setting: organizations can reshuffle people, projects, and search direction without producing a uniform short-run collapse in headcount.

Third, the heterogeneity tools repeatedly point to baseline size as a central moderator, with deal intensity relative to firm size mattering more on the target side. That is a much better economic statement than “M&A hurts innovation on average.” It says that organizational capacity and shock intensity shape the post-merger response.

### 3. The current uploaded firm results favor the following interpretation
- **Acquirors:** weak average patent and citation effects, some evidence of reduced exploration, and suggestive evidence that mobility and staffing respond more than output. The acquiror side looks more like organizational reshuffling than like a sharp productivity collapse.
- **Targets:** stronger negative baseline effects for patents, citations, exploration, and inventor count, but with enough pre-trend and estimator-sensitivity concerns that these should be presented as **evidence of disruption**, not as a settled causal average treatment effect.
- **Across both sides:** placebos and advanced estimators imply that firm-level output coefficients should be treated cautiously, while heterogeneity and mechanism-style outcomes are more informative.

### 4. The inventor panel remains the mechanism side of the project
The inventor sections already in the notebook should remain prominent. Even before tomorrow's remaining uploads, the structure of the current results supports the broader idea that the project is most interesting when it links firm reorganization to inventor behavior, search direction, and mobility rather than trying to force all value through one aggregate patent equation.


## Step M. How I would present the strongest results in the final notebook

Based on the current uploaded results, I would recommend presenting the findings in the following hierarchy.

### Headline finding
**M&A appears to reshape innovative organization more clearly through heterogeneous disruption, inventor reallocation, and changes in search behavior than through one uniform average effect on firm-level patent output.**

That sentence is careful, but it is still strong. It tells the reader that the project identifies a meaningful mechanism while respecting the limits of the average-effect evidence.

### Supporting finding 1
**Acquiror-side evidence is most consistent with reorganization rather than collapse.**

Average patent and citation effects for acquirors are weak across advanced estimators, but exploration and inventor-flow outcomes suggest post-merger reshuffling. This is a plausible economic pattern: large organizations may absorb the deal without an immediate sharp drop in counts, while still changing how inventive labor is allocated.

### Supporting finding 2
**Target-side baseline declines are economically large, but not estimator-proof.**

For targets, the baseline matched TWFE specifications show sizeable post-merger weakening in patents, citations, exploration, and inventor counts. However, large positive leads in event studies, disagreement across Sun-Abraham and BJS, and significant placebo effects in some specifications all mean the notebook should frame these results as evidence of disruption rather than as a fully settled causal decline.

### Supporting finding 3
**Heterogeneity is central, not peripheral.**

The repeated importance of baseline size and related financial characteristics suggests that the effect of M&A depends on absorptive capacity and shock intensity. This is exactly the sort of conclusion that is valuable to a tech economist or applied research team: it moves the analysis beyond a simple average treatment effect and toward organizational mechanism.

### Supporting finding 4
**The project gains credibility by showing where the simple design fails.**

The placebo results are not an embarrassment. They are part of what makes the notebook persuasive. They show that the analysis does not equate “one significant coefficient” with “truth,” and they justify the decision to emphasize cross-estimator agreement, mechanism outcomes, and heterogeneity rather than isolated baseline significance.

That ordering is important. It makes the project sound like a serious innovation-economics paper rather than an overclaimed “M&A kills innovation” story. For recruiting purposes, that is an advantage: the project reads as careful, mechanism-aware, and empirically disciplined.


## Step N. What the notebook should say explicitly about limitations

A strong public notebook should make the following limits visible.

### 1. Public-firm scope
The analysis follows inventors only when they can be linked to publicly listed firms. That improves observability but means the mobility results are not the full universe of inventor moves.

### 2. Matching improves design but does not prove exogeneity
The matched stacked design is a strong improvement over a raw pooled comparison, but it does not eliminate all concerns about differential pre-trends or event selection.

### 3. Firm-level output results are estimator-sensitive
This is now a direct result of the uploaded outputs, not just a generic caveat. For several patent and citation outcomes, the baseline TWFE DiD is economically large, but the event-study leads are not flat, Sun-Abraham and BJS sometimes disagree materially, and placebo timing can generate significant effects. The notebook should therefore say plainly that **firm-level output effects are informative but not fully settled**.

### 4. Staggered-adoption methods help, but they are still design-based estimators
Sun-Abraham, BJS, and CSDID reduce known TWFE problems, but they do not make interpretation mechanical. One still has to inspect support, pre-trends, estimator agreement, and whether the effect survives placebo scrutiny.

### 5. Advanced inventor methods are run selectively
This is a practical choice to keep the workflow feasible, but it means the most computationally intensive cross-checks are not literally estimated on every row of the full inventor panel.

### 6. Output effects are less settled than mechanism effects
The project should not oversell patent counts or citations if those outcomes are less stable across methods. The more credible parts of the current notebook are the mechanism-side responses, the cross-sectional heterogeneity, and the disciplined interpretation of estimator disagreement.

### 7. Some firm-level auxiliary outcomes are not estimable in the full FE specification
A subset of the firm-level `avg_rel_*` inventor-flow quality measures and a few target-side auxiliary outcomes fail rank checks in the matched stacked regressions. The added diagnostics indicate that this is not a simple coding error: the raw regressor matrix remains full-rank, but the relevant outcome-specific samples are thin and the ratio-style variables are extremely skewed and denominator-sensitive. The most defensible interpretation is therefore that these failures reflect **estimability limits of selected auxiliary variables after fixed-effects absorption**, not a general flaw in the design.

Writing these caveats directly into the notebook will improve trust rather than weaken the project.


## Summary of the analysis

The analysis workflow is one of the strongest parts of the project.

It combines:

- a careful **firm-level matched stacked event-study design**,
- an **inventor-level mechanism design** built around matched control pseudo-events,
- multiple **staggered-adoption robust estimators**,
- disciplined **heterogeneity analysis** using baseline `Z` variables,
- and a meaningful set of **placebo and balance diagnostics**.

Just as important, the notebook now shows that the project does not treat all statistically significant outputs as equal. It distinguishes between:
- baseline screens,
- robustness checks,
- placebo failures,
- and results that are actually persuasive enough to headline.

That is exactly how a strong applied research project should read.

For the evidence currently uploaded, the firm-level takeaways are best summarized as follows:

- The baseline firm DiD produces economically meaningful negative effects for several **target-side** output measures, especially patents, citations, exploration, and inventor counts.
- Those target-side output effects are **not** fully stable across estimators. Event-study leads, BJS disagreement, and placebo significance imply that they should be interpreted as evidence of disruption rather than as fully settled causal averages.
- The **acquiror** side shows much weaker average patent and citation effects. The more plausible interpretation is post-merger **reorganization**: exploration, mobility, and staffing margins adjust more visibly than aggregate output.
- Heterogeneity matters. The recurring importance of baseline size and related financial capacity variables suggests that absorptive capacity is central to the post-merger response.
- Some auxiliary firm outcomes fail under the full FE design because they are sparse and denominator-sensitive, which is an estimability issue rather than evidence of a broken pipeline.

For recruiting and external readers, this is a strong story precisely because it is disciplined. The notebook does not claim more than the evidence supports. Instead, it shows how to move from a noisy set of event-study outputs to a defensible economic interpretation about organizational adjustment, inventor reallocation, and heterogeneous innovation disruption after M&A.


## Empirical design in plain language

The project uses publicly listed firms and patent-linked inventors to study what happens around merger and acquisition events.

At a high level, there are two complementary designs:

### 1. Firm-year panel
This asks whether treated firms differ from comparable untreated firms after a deal. It is useful for aggregate outcomes such as:
- patent counts,
- citations,
- novelty and exploration measures,
- inventor flows in and out of the firm.

### 2. Inventor-year / inventor-event panel
This asks whether inventors attached to treated firms behave differently after a deal relative to matched controls. This is especially useful for:
- mobility,
- exploration,
- inventor-level productivity,
- within-firm heterogeneity.

The value of the project is that it combines both levels:
the firm panel captures organizational outcomes, while the inventor panel helps interpret the mechanism.

## Identification strategy

The main estimators are:
- baseline fixed-effects DiD,
- event studies,
- Sun–Abraham interaction-weighted event studies,
- Borusyak–Jaravel–Spiess imputation,
- selected matching and stacked-panel designs.

The reason for using multiple estimators is straightforward: staggered treatment timing can make simple two-way fixed-effects event studies misleading, especially when treatment effects evolve over time or differ across cohorts.

That is why, in interpretation, this notebook gives more weight to:
- patterns that survive across multiple estimators,
- results without severe pre-trend problems,
- economically sensible magnitudes and shapes.

It gives less weight to:
- isolated significant coefficients,
- event studies with visibly unstable pre-trends,
- very large or implausible coefficients,
- patterns that reverse sign across methods without a clear explanation.

In [ ]:
# Core imports for the notebook
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.3f}".format)

# Repository root: adjust after cloning if needed
REPO_ROOT = Path(".").resolve()

# Example output folders used by the analysis script
MODEL_OUT = REPO_ROOT / "Model_outputs"
PLOTS_OUT = MODEL_OUT / "Plots"

print("Repository root:", REPO_ROOT)
print("Model outputs:", MODEL_OUT)

## Step 1. Load the key outputs

The notebook is easiest to use if the heavy analysis has already been run and the output CSVs / plots already exist.

The helper file added to the repository can build:
- summary statistics tables,
- panel diagnostics,
- robustness scorecards,
- compact headline tables for the notebook.

The next cell assumes those helper functions are available.

In [ ]:
# These imports assume the helper file from this draft has been added to the repo.
# If the file is placed elsewhere, adjust the import path.
try:
    from notebook_additions_minimal import (
        build_headline_scorecard,
        summarize_firm_panel,
        summarize_stacked_panel,
        summarize_inventor_event_panel,
        build_result_inventory,
        write_notebook_support_outputs,
    )
    HELPERS_AVAILABLE = True
except Exception as e:
    HELPERS_AVAILABLE = False
    print("Helper import failed:", e)

## Step 2. Recommended summary statistics to include

A public-facing notebook should show basic descriptive structure before jumping into causal estimates.

I recommend including four compact tables:

1. **Firm panel summary**  
   Number of firms, years, treated firms, and baseline means for headline outcomes.

2. **Stacked firm-panel summary**  
   Number of treated and control firm-event units, balance over the event window, and treated share.

3. **Inventor-event panel summary**  
   Number of inventors, inventor-event units, treated and control units, and event-window balance.

4. **Headline baseline means**  
   Means at `t = -1` for the outcomes that appear in the narrative.

This helps readers anchor effect sizes and see the scale of the analysis.

In [ ]:
# Example usage once the relevant dataframes are loaded in memory:
#
# firm_summary = summarize_firm_panel(firm_panel, outcomes=["xi_real", "arriving_inventors_count", "exploration_firm"])
# acq_stack_summary = summarize_stacked_panel(stacked_acquiror, label="Acquiror stack")
# tgt_stack_summary = summarize_stacked_panel(stacked_target, label="Target stack")
# inv_summary = summarize_inventor_event_panel(inv_es, label="Inventor event panel")
#
# display(firm_summary)
# display(acq_stack_summary)
# display(tgt_stack_summary)
# display(inv_summary)

## Step 3. How to think about the results

A good empirical reader should evaluate the results in layers rather than asking whether one coefficient is statistically significant.

### A. What is a true headline result?
A result is headline-worthy if most of the following are true:
- the sign is stable across reasonable specifications,
- event-study pre-trends are not obviously problematic,
- the magnitude is economically interpretable,
- the estimate aligns with the mechanism story,
- more robust staggered-treatment estimators do not overturn it,
- and placebos do not generate similarly strong patterns.

### B. What is a supporting result?
A result is useful but secondary if:
- it supports the mechanism,
- it fits the economic story,
- but it is weaker, noisier, or more sensitive than the main result.

### C. What should stay out of the headline?
Results should stay out of the headline if:
- they are driven by a single estimator,
- they rely on implausibly large coefficients,
- they have visibly non-flat pre-trends,
- they reverse sign across methods without a clear explanation,
- or they are only present in a downsampled advanced run and not in the full baseline panel.

This project is strongest when interpreted with that discipline.



## Executive synthesis of the empirical results: firm panel and inventor-year panel together

Before turning to the estimator-by-estimator discussion, it helps to state the overall message of the project in plain research terms.

At a high level, the evidence points to **post-merger disruption with strong heterogeneity**, not to one clean universal average treatment effect. That statement is true in both the **firm panel** and the **inventor-year panel**, although the two panels illuminate different margins.

### 1. What the firm panel says

The firm panel answers the aggregate question: **how does the innovative output of treated firms evolve after M&A relative to matched controls?**

Across the baseline stacked DiD specifications, the first-pass picture is fairly negative. Both acquirors and targets often show weaker post-treatment patenting, citation-based outcomes, and innovation-value measures. That is exactly the kind of pattern one would expect if mergers interrupt project pipelines, create internal overlap, delay decisions, or trigger selective departure of inventors and managers.

But the more advanced estimators make clear that the average-effect story is **not equally strong for every outcome**:

- For **acquirors**, the more convincing average-effect evidence is concentrated in outcomes such as **cited patents** and some related patent-composition measures, rather than in every patent bucket or every inventor-flow margin.
- For **targets**, the average-effect results tend to look negative as well, but the uploaded advanced evidence is thinner and sometimes noisier than for acquirors.
- The most stable pattern in the firm panel is **heterogeneity**. Larger acquirors appear more resilient, while on the target side the strongest negative effects tend to be associated with **larger deals relative to the target**.

Economically, that is a strong and intuitive result. A merger is not just a scale shock. It is an organizational shock. Firms with deeper managerial capacity, broader internal labor markets, and more slack are better able to absorb it. Smaller or more exposed firms are more likely to see integration frictions translate into weaker observed innovation.

### 2. What the inventor-year panel says

The inventor-year panel answers a different question: **how do individual inventors associated with treated firms change their behavior and outcomes after the deal?**

This is not just a smaller-scale version of the firm panel. It is a mechanism panel. It tells us whether the post-merger shock looks like:

- lower inventor productivity,
- greater inventor mobility,
- a shift from exploitation toward exploration,
- selective retention of some inventors but not others,
- or simple reallocation across project types.

That distinction matters. A firm can lose aggregate output even when some inventors become more exploratory, and a firm can preserve inventor-level activity while still losing organization-level efficiency because coordination breaks down.

The baseline inventor-year results suggest a **sharp contrast between acquirors and targets**:

- For **acquirors**, the baseline results look like **reorganization**. Inventors appear to become more exploratory, less exploitative, slightly more novel, and somewhat more likely to move, while measures such as `xi_real` deteriorate.
- For **targets**, the inventor-year baseline looks more uniformly negative: weaker patents, weaker citations, lower novelty, lower `xi_real`, and weaker move-related outcomes.

That contrast is economically plausible. Acquiror inventors remain inside the surviving organization and may be reallocated across technologies, teams, and product lines. Target inventors are more likely to face dislocation, loss of fit, or reduced access to the organizational resources that supported their pre-deal productivity.

### 3. Why the firm and inventor panels do not have to match one-for-one

It would actually be surprising if the firm panel and inventor panel were numerically identical in sign and magnitude.

There are several reasons:

1. **Aggregation changes the object being measured.**  
   Firm outcomes combine many inventors, units, and continuation choices. Inventor outcomes isolate individual behavior.

2. **Selection matters.**  
   Post-merger firms may retain some inventors, lose others, and reassign others. The firm panel combines those margins; the inventor panel partially separates them.

3. **Innovation is multi-dimensional.**  
   Patent counts, citations, novelty, and `xi_real` are not the same object. A merger may raise search and recombination while lowering realized value in the short run.

4. **Timing matters.**  
   Inventor behavior can adjust quickly. Firm-level citation outcomes adjust slowly because they are tied to filing, grant, and citation lags.

### 4. Econometric intuition for why the results are mixed across estimators

The variation across estimators is not a flaw by itself. In this setting, some disagreement is exactly what one should expect.

- **Stacked DiD** is a useful benchmark, but in staggered-adoption settings with dynamic effects it can still blur timing heterogeneity.
- **Sun–Abraham** is especially useful for seeing whether apparent post-treatment effects are partly reversals of pre-trends.
- **BJS** is attractive because it is robust to treatment-effect heterogeneity under its identifying assumptions, but it is still sensitive to support and to how outcomes are measured.
- **CSDID** is valuable for dynamic aggregation in staggered settings, but in these inventor-year outputs some of its estimates are extremely precise and occasionally at odds with both baseline and SA, which is a sign to interpret them as informative rather than automatically decisive.
- **SCM** can be appealing for selected firm outcomes, but successful fit counts matter, and in sparse patent outcomes the subset of units with usable donor fits can differ meaningfully from the full treated sample.
- **Causal forests** are not delivering sharp average treatment effects here, but they are useful because they repeatedly identify baseline firm size and related financial characteristics as important moderators.

In other words, the right use of the estimator set is not “vote counting.” It is triangulation.

### 5. What results should anchor the notebook

If the notebook is meant to be clear, persuasive, and useful for recruiting, the results hierarchy should be:

**Headline results**
- M&A is associated with meaningful post-merger innovation disruption.
- The disruption is **heterogeneous**, not uniform.
- Larger acquirors appear more resilient.
- Targets in relatively larger deals appear more exposed.
- On the inventor side, acquirors look reorganized; targets look more clearly disrupted.

**Supporting mechanism results**
- Acquiror inventor-year outcomes suggest a shift toward exploration and away from exploitation.
- Acquiror patent-composition results suggest that externally recognized patent output may deteriorate more clearly than raw counts alone imply.
- Mobility outcomes point to reallocation and selective reshuffling, though they are not equally stable across methods.

**More fragile or secondary results**
- Outcomes with mixed signs across advanced estimators,
- outcomes with evident placebo sensitivity,
- and outcomes based on narrow patent buckets or short-support dynamic cells.

That framing makes the project stronger, not weaker. It shows that the analysis is taking both economics and identification seriously.



## Results discussion based on the exported outputs in this upload

This section now reflects the broader set of files provided across both uploads:

- baseline and triple-DiD significance summaries for **acquiror** and **target** firms,
- selected **Borusyak–Jaravel–Spiess (BJS)** event-study outputs,
- selected **Sun–Abraham (SA)** event-study outputs for acquiror outcomes,
- selected **stacked synthetic control (SCM)** summaries,
- selected **causal forest (CF)** summaries,
- and **CSDID feasibility diagnostics** for selected acquiror outcomes.

The goal of this discussion is not to force every estimator into a single sign. The stronger goal is to identify which findings are **stable enough to anchor the main story**, which findings are useful as supporting mechanism evidence, and which findings should be presented more cautiously.

A core theme running through the outputs is that this is **not** a setting where one should expect every estimator and every outcome to line up perfectly. Firm-level patenting is lumpy, citations arrive with delay, many outcomes are zero-heavy, and mergers can alter both real innovative activity and the internal organization of patent production. That makes this an excellent setting for a serious applied-economics notebook: the right task is to separate **robust structure** from **estimator-sensitive averages**.

Two broad conclusions remain the safest and strongest.

1. The evidence does **not** support a simplistic universal claim such as “M&A always lowers innovation by X percent.”
2. The evidence **does** support a richer conclusion: merger effects are heterogeneous, and much of the action appears to come through **integration capacity, portfolio reorganization, and differential ability to absorb disruption**.



## What looks strongest in the current evidence

### 1. Baseline DiD still provides the useful first-pass picture
The baseline stacked DiD continues to suggest that post-merger periods often look weaker on standard innovation outcomes.

For **acquirors**, the significant baseline post-treatment effects include negative coefficients for:
- `log1p_total_patents`,
- `log1p_cited_patents`,
- `log1p_uncited_patents`,
- `log1p_backward_cites`,
- `log1p_xi_real`,
- `log1p_self_cites`,
- `log1p_top10_to_2_patents`,
- `log1p_arriving_inventors_count`,
- `log1p_departing_inventors_count`,
- and `log1p_sum_patents_pre_move_arrivals`.

For **targets**, the significant baseline post-treatment effects include negative coefficients for:
- `log1p_total_patents`,
- `log1p_cited_patents`,
- `log1p_cites`,
- `log1p_backward_cites`,
- `log1p_xi_real`,
- and `avg_rel_exploration_departures`.

This remains useful because it says that, before we get sophisticated, the data already look consistent with **post-deal disruption**. In firm-level innovation data, that is economically plausible. Mergers can interrupt project pipelines, create overlapping R&D teams, trigger inventor exits, delay internal decision-making, and redirect managerial attention away from exploratory activity.

At the same time, baseline DiD is not the end of the story. In staggered-treatment settings with noisy firm outcomes, a strong notebook should immediately ask:
- do more robust event-study estimators tell a similar story,
- do the signs hold for the more interpretable outcomes,
- and do the patterns fit an economic mechanism rather than a purely statistical artifact?

That is where the uploaded SA, BJS, SCM, and CF results become especially informative.


## The best public-facing headline: heterogeneity is the main result, not a uniform average effect

The firm-level results are now sufficiently developed to support a disciplined headline: **the central result is heterogeneity in post-merger innovation performance, not one uniform average effect of M&A.** The average firm-level effects are often negative or mixed, but the economically clearer and more reproducible pattern is that merger effects depend strongly on the kind of firm absorbing the shock.

### Acquirors: baseline size is the strongest and most coherent moderator

For acquirors, the most convincing heterogeneity dimension is still **baseline firm size**, proxied by pre-deal log sales. The newly uploaded baseline `Z_log_sales_q3` files reinforce rather than weaken that conclusion. In the low-sales tercile, the baseline post-merger coefficients are often negative or close to zero for core outcomes, while the interaction terms for larger terciles are frequently positive for economically important margins such as `log1p_total_patents`, `log1p_cites`, `log1p_cited_patents`, `log1p_uncited_patents`, `log1p_arriving_inventors_count`, and `log1p_xi_real`. The broad pattern is therefore not that large acquirors experience large positive post-merger gains in absolute terms, but that they appear **substantially more resilient than smaller acquirors when integration begins to disrupt innovation**.

That interpretation is economically intuitive. Larger acquirors likely have:
- deeper managerial capacity to handle integration,
- more diversified R&D portfolios,
- stronger internal labor markets,
- more slack to preserve projects during restructuring,
- and greater absorptive capacity when teams, technologies, and reporting lines are recombined.

In that sense, scale does not eliminate merger friction, but it appears to **attenuate the damage**. This is exactly the kind of heterogeneity pattern that is more interesting than a single average-treatment estimate, because it points to a concrete organizational mechanism.

### Targets: deal intensity matters more than simple size

For targets, the most compelling heterogeneity result remains **deal value relative to target scale**. The notebook already emphasized this, and nothing in the later uploads overturns it. The target-side story is therefore naturally complementary to the acquiror-side story:

- for **acquirors**, what matters most is baseline organizational capacity;
- for **targets**, what matters most is how transformative the transaction is relative to the target itself.

That asymmetry is economically plausible. Acquirors are the integrating organizations, so baseline managerial and technological depth should matter most on their side. Targets, by contrast, are more likely to experience disruption in proportion to how large and consequential the deal is relative to their own scale.

### Why the contrast with relative deal value for acquirors is substantively useful

The newly uploaded acquiror specifications based on **relative deal value** remain much weaker overall. Outside of a few organizational margins such as inventor inflows, headcount-related measures, and rolling self-similarity, they do not show a broad or stable gradient across core patent or citation outcomes. That contrast is substantively useful. It suggests that on the acquiror side, the key issue is not simply whether the deal is large relative to the balance sheet or market value. The more important question is whether the acquiror has the **organizational capacity to absorb and govern the integration shock**.

So the firm-level public-facing headline can now be stated more sharply: **M&A effects are heterogeneous in a way that maps naturally into organizational economics. Large acquirors are better able to absorb disruption, while larger relative deals are more disruptive for targets.**


## How the advanced estimators change the interpretation

The additional files sharpen the interpretation in two ways.

### A. Average effects are real enough to be interesting, but not always stable enough to be the sole headline
For the outcomes where more robust estimators were uploaded, the signs are often suggestive but not perfectly aligned across methods.

#### Acquiror inventor-flow outcomes
For **arriving inventors**:
- baseline DiD is negative,
- BJS is negative in post years 1 through 3 and remains negative later, though less precise,
- SCM is near zero overall,
- and the causal forest ATE is positive but very imprecise.

For **departing inventors**:
- baseline DiD is negative,
- BJS is positive but imprecise,
- SCM is negative and statistically significant,
- and the causal forest ATE is again imprecise.

The right interpretation is not that one method is “wrong” and another “right.” These outcomes are measuring a reorganization margin that can move in complicated ways. A merger can simultaneously reduce net inflows, selectively retain key staff, and create exits among overlapping or lower-priority teams. Depending on the identifying comparison and aggregation, different methods can emphasize different pieces of that adjustment.

#### Target citations
For **target citations**, the cross-method disagreement remains substantial:
- baseline DiD is negative,
- BJS is positive in the uploaded post-treatment years,
- SCM is negative and only borderline precise,
- and the causal forest ATE is close to zero with a very wide interval.

This is exactly the kind of outcome where caution is warranted. Citation outcomes are particularly noisy because they are affected by both the flow of new patents and the ex post citation process, which arrives with delay and can be influenced by portfolio composition, not just contemporaneous research productivity.

### B. The patent-composition outcomes for acquirors add a useful new layer
The newly uploaded acquiror files make the story richer. They suggest that the post-merger response is not only about “more or fewer patents,” but also about **what kind of patents and citation activity move most clearly**.



## New insight from the additional acquiror files: composition matters, not just counts

The new acquiror outputs for `cited_patents`, `top10_to_2_patents`, `uncited_patents`, and `self_cites` are especially helpful because they speak to **composition of innovative output**.

### 1. Cited patents look more consistently negative than several other patent components
For **acquiror cited patents**:
- baseline DiD is negative,
- BJS is negative in every uploaded post-treatment year and statistically meaningful early on,
- SCM is clearly negative and statistically significant overall,
- while the causal forest ATE is imprecise.

This is one of the cleaner pieces of evidence that post-merger disruption may be concentrated in **patents that attract downstream attention**, rather than only in raw patent volume.

A reasonable economic interpretation is that integration shocks disproportionately affect the kinds of projects that require coordination, continuity, and time to mature into influential patents. Cited patents are often produced by stronger projects or more effective internal execution. Those may be exactly the projects most vulnerable to disruption when teams are restructured or decision rights are reassigned.

### 2. Mid-tier cited patents also lean negative
For **top10_to_2_patents**, the evidence is somewhat weaker than for cited patents overall, but still leans negative:
- baseline DiD is negative,
- SA drifts negative after treatment,
- SCM is negative and statistically significant,
- BJS is also negative though not very precise,
- and the causal forest ATE is imprecise.

This is useful because it suggests the pattern is not confined only to the most extreme citation tail. It looks more like a broader weakening in the distribution of externally recognized patent output.

### 3. Uncited patents are harder to interpret and probably noisier
For **uncited patents**, the methods disagree more:
- baseline DiD is negative,
- SA turns clearly negative after treatment and becomes more negative over event time,
- SCM is strongly negative,
- but BJS is positive though imprecise,
- and the causal forest ATE is near zero and imprecise.

This is not surprising. Uncited patents are a particularly noisy object. They may represent genuinely low-impact inventions, patents that have not had time to accumulate citations, defensive filings, or simply sparse outcomes in firm-years with many zeros. Because of that, uncited-patent results should be treated as **supportive texture**, not the center of the story.

### 4. Self-citations are conceptually ambiguous
For **self-cites**, the estimates are again mixed:
- baseline DiD is negative,
- SA becomes negative in later post years,
- SCM is negative overall,
- but BJS turns positive and grows over post-treatment years, though imprecisely,
- and the causal forest ATE is positive but wide.

This is a case where economic interpretation matters a lot. Self-citations can fall if integration disrupts internal knowledge continuity. But they can also rise if the combined firm consolidates overlapping technological lines and reuses its own portfolio more intensively. In other words, self-cites are not a clean “good” or “bad” innovation measure. They partly reflect how the merged firm reorganizes internal technological inheritance. That is exactly why they should be discussed, but not used as a headline metric.


## Additional discussion: acquiror triple-DiD event studies by baseline sales terciles

The newly uploaded files add an important dynamic layer to the **acquiror-side triple-DiD event studies** that interact treatment timing with baseline sales terciles, `Z_log_sales_q3`. These results are worth discussing explicitly because they show why the heterogeneity story is stronger than the pooled event-study story.

### How to read these event-study tables

In these specifications, the coefficients `rt_k` trace the dynamic treatment effect for the **omitted sales tercile**, which is most naturally interpreted as the **lowest baseline sales tercile**. The interaction terms `rt_k__Z_log_sales_q3_1` and `rt_k__Z_log_sales_q3_2` are deviations for the middle- and high-sales terciles relative to that omitted low-sales group.

So for any event year `k`:

- `rt_k` is the effect for the low-sales tercile,
- `rt_k + rt_k__Z_log_sales_q3_1` is the effect for the middle-sales tercile,
- `rt_k + rt_k__Z_log_sales_q3_2` is the effect for the highest-sales tercile.

That is crucial. The raw `rt_k` path by itself can make the dynamics look more negative than they are for larger acquirors, because it shows only the lowest-sales firms directly.

### High-level pattern across the uploaded acquiror files

Across the uploaded sales-tercile event studies, a common pattern emerges.

First, the **lowest-sales acquirors** often display the most problematic dynamics. In several outcomes they have positive pre-treatment coefficients relative to the omitted `t = -1` period and then flatten or drift negative after treatment. This is especially visible in outcomes such as `log1p_cites`, `log1p_total_patents`, and `rolling_self_similarity`. That means the lowest-sales firms are precisely where pre-trend concerns are most pronounced and where post-merger weakening appears strongest.

Second, the **middle- and high-sales terciles usually look better than the low-sales tercile**, often substantially so. The interaction terms are frequently positive for core outcomes, especially for the highest tercile, implying that larger acquirors experience a materially smaller deterioration than small acquirors and in some cases remain close to zero on net.

Third, the dynamic evidence is most persuasive for a **relative-resilience interpretation**, not for a clean “big firms improve after mergers” claim. The event-study paths do not show a universal post-merger boom among large acquirors. Rather, they show that the adverse dynamic visible among smaller acquirors is **muted for larger acquirors**.

### What this adds beyond the pooled baseline event studies

This is why the triple-DiD event studies are so valuable. The pooled event studies average together firms with very different capacities to absorb disruption. Once the sample is allowed to vary systematically with baseline scale, the firm-level results become easier to interpret:

- small acquirors look more vulnerable to integration-related disruption,
- large acquirors look more capable of preserving innovative activity,
- and the resulting average effect is an amalgam of those very different paths.

That is a more publication-ready interpretation than simply reporting that “the average post coefficient is negative.” It links the empirical pattern to a concrete economic idea: **absorptive capacity and organizational depth shape whether a merger becomes mainly disruptive or merely manageable.**

## Additional discussion: acquiror triple-DiD by relative deal value

The newly uploaded **acquiror-side triple-DiD specifications by relative deal value**, `Z_deal_rel_q3`, are useful mainly because they clarify what the data are **not** strongly saying. In contrast to the sales-based heterogeneity results for acquirors, these specifications do **not** generate a broad, stable pattern across patent counts, citation measures, or the exploration/exploitation outcomes. That negative result is itself informative. It suggests that, on the acquiror side, **baseline organizational capacity and scale** matter more consistently than deal size relative to the acquiror’s own market value.

The newly uploaded **baseline results for the same `Z_deal_rel_q3` interaction** reinforce that interpretation rather than changing it. Across the baseline files, the relative-deal-value interaction remains mostly weak for the core patent and citation outcomes. The few suggestive margins are concentrated in organizational or mechanism-style outcomes such as inventor arrivals, selected mobility-quality measures, and rolling self-similarity. Even the level specifications for `exploration_firm` and `exploitation_firm` remain close to zero. So the broader conclusion is now more secure: this is not simply a feature of one transformed outcome or one event-study variant. In the acquiror sample, **relative deal value is not emerging as a strong general moderator of post-merger innovation performance**.

### How to read these triple-DiD tables

As in the other grouped heterogeneity specifications, the omitted category is most naturally interpreted as the **lowest relative-deal-value tercile**. Under that coding:

- `Post_Treated` is the post-merger treatment effect for the omitted low relative-deal group,
- `Post_Treated_Z_deal_rel_q3_1.0` is the incremental post effect for the middle tercile relative to that omitted group,
- `Post_Treated_Z_deal_rel_q3_2.0` is the incremental post effect for the highest tercile relative to that omitted group.

### What the results imply

The most natural reading is therefore comparative. For acquirors, the data are not telling a story in which “bigger relative deals uniformly hurt more” or “bigger relative deals uniformly help more.” Instead, the relative-deal-value split seems to matter only sporadically, and mostly on secondary margins. That is substantively useful because it sharpens the main claim:

- **acquiror heterogeneity is primarily about absorptive capacity and organizational scale**;
- **target heterogeneity is more about deal intensity relative to the target**.

That asymmetry gives the project a cleaner economic interpretation than a symmetric deal-size story would. It suggests that the burden of integration is not governed by the same variable on both sides of the transaction.

## Additional discussion: baseline acquiror event-study results and how they compare with the triple-DiD specifications

The newly added baseline event-study outputs are useful because they answer a different question from the sales-heterogeneity specifications. The baseline models ask whether, **on average**, acquiror firms exhibit systematic changes around the deal. The triple-DiD models then ask whether those pooled average effects conceal meaningful differences across acquirors with different baseline sales.

### 1. What is most stable across the baseline firm-panel event studies?

The baseline results now point to a fairly consistent qualitative pattern, but one that needs to be stated carefully. On average, acquiror firms do **not** show a dramatic immediate collapse at the event year. Instead, the most common pattern is:

- modestly elevated or flat coefficients in the pre-period,
- little clear break exactly at treatment,
- and a gradual move toward weaker outcomes in the later post years for several patent and citation measures.

That pattern is especially visible in `log1p_cites`, `log1p_total_patents`, `log1p_uncited_patents`, and `rolling_self_similarity`. In each of these cases, the later post-treatment years look weaker than the omitted pre-period, although the timing and precision differ across outcomes.

This is economically plausible. Integration frictions need not show up immediately in annual patent production. Some projects are already in the pipeline when the merger closes, while the effects of reorganization, reprioritization, and team disruption can accumulate over several years.

### 2. What do the baseline event studies say about pre-trends?

The baseline event studies also make clear that **pre-trends are not equally clean across outcomes**.

For `log1p_cites` and `log1p_total_patents`, the pre-period coefficients are often positive relative to `t = -1`, which means the pooled event studies should not be read as fully clean causal break designs. Part of the post-merger decline may reflect reversal from a stronger pre-treatment path rather than a sharp treatment-onset effect. The cited-patent outcomes show a similar issue, though in somewhat weaker form.

By contrast, outcomes such as `log1p_arriving_inventors_count` and `rolling_self_similarity` are somewhat more informative as mechanism variables. Their pre-periods look flatter, and the post-treatment weakening appears later and more gradually. Those are still not perfect designs, but they are easier to interpret as post-merger adjustment rather than simple continuation of a pre-existing trend.

### 3. How the baseline event studies and the sales-tercile triple-DiD fit together

Taken alone, the pooled baseline event studies are suggestive but not ideal headline evidence because several important outcomes show noticeable lead patterns. But once combined with the sales-tercile results, their role becomes clearer.

The pooled event studies tell us that there is **some average post-merger weakening**, especially in later years and especially for patent and citation outcomes. The triple-DiD event studies then explain why the pooled picture looks noisy: it is averaging together small acquirors that appear relatively vulnerable and large acquirors that appear substantially more resilient.

So the baseline event studies are still useful, but primarily as a descriptive first layer. The more publication-ready interpretation comes from reading them **together** with the heterogeneity results:
- pooled average effects are mixed and sometimes contaminated by pre-trend concerns,
- but the heterogeneity pattern is economically coherent,
- and it consistently points toward organizational capacity as the key moderator on the acquiror side.


## Why some results look mixed: economic and measurement intuition

The variation across outcomes and estimators is not just a statistical nuisance. Much of it is economically understandable.

### 1. Firm-level patenting is lumpy and sparse
Patenting is not smooth firm-year production. Even large public firms can have substantial year-to-year variation in patent counts, citation-weighted measures, and specialized patent buckets. That creates:
- many zeros in narrower outcomes,
- sensitivity to a small number of important projects,
- and greater estimator disagreement when comparisons rely on different sets of treated-control matches or different weighting schemes.

This is especially relevant for outcomes like `top10_to_2_patents`, `uncited_patents`, and `self_cites`, where small changes in a few patents can move the firm-year total considerably.

### 2. Citation outcomes arrive with lag
A patent filed in the post-treatment period does not reveal its citation profile immediately. That means citation-based outcomes can be contaminated by timing mismatch:
- some post-merger patents have not yet had enough time to accumulate citations,
- some observed citations reflect pre-merger projects that continue to mature,
- and event-time estimates can therefore blend project-timing effects with real treatment effects.

That timing issue helps explain why citation outcomes can differ more across methods than raw patent counts.

### 3. Mergers often change organization before they change measured output
The earliest merger effects may show up in:
- staffing,
- project selection,
- budget control,
- reporting lines,
- and portfolio rationalization,

before they show up in total patent output. That is one reason the heterogeneity results are so informative. They suggest that **who can absorb the organizational shock** matters at least as much as the average mechanical effect on patent counts.

### 4. Acquiror and target outcomes need not mirror each other
The acquiring firm is usually integrating new assets into a larger existing platform, while the target is being integrated into another organization. Those are very different adjustment problems. So it is entirely plausible that:
- acquiror results are moderated mostly by the acquiror’s scale and capacity,
- while target results are moderated more by how transformative the transaction is for the target.

### 5. Small-sample issues can matter even when the raw panel looks large
Some of the outcome-specific advanced-method samples are effectively much smaller than the full panel might suggest.
- SCM uses only treated units with acceptable pre-treatment fit, so successful fits can be a modest share of eligible treated firms.
- Dynamic estimators thin out at longer horizons as the number of usable treated cohorts shrinks.
- Narrower patent outcomes magnify this problem because the signal-to-noise ratio worsens.

So even with a reasonably large panel, some event-time or outcome-specific results can still be precision-limited.


## Diagnostic note on the firm-level specifications that fail rank checks

The additional diagnostic files make the interpretation of the firm-level estimation failures much clearer, and they suggest that these failures should be treated as a **limited support / estimability issue for selected auxiliary outcomes**, not as a general problem with the firm analysis.

### 1. The problematic `avg_rel_*` arrival and departure outcomes are empirically fragile by construction

The strongest conclusion from the diagnostics is that the relative arrival and departure measures are exactly the kinds of variables one would expect to create unstable estimation samples. These outcomes combine:
- many zeros,
- mass points at or near zero,
- ratios that can become very large when denominators are small,
- and extreme right tails that are orders of magnitude larger than the central part of the distribution.

That pattern is especially stark for the arrival-based relative measures. In several cases the median is essentially zero while the 99th percentile is extremely large, and the maximum is larger still by many orders of magnitude. The diagnostics therefore support the interpretation that these variables are not simply “noisy” in the ordinary sense; they are **pathological ratio outcomes** whose scale is highly sensitive to sparse or mechanically small denominators.

This matters for interpretation. When a specification uses firm fixed effects, year fixed effects, lagged controls, and then conditions on the non-missing sample for one of these fragile ratio outcomes, the effective estimating sample can become both thin and numerically awkward. In that setting, failure of the rank check is not evidence that the underlying baseline design is wrong. It is better interpreted as evidence that some auxiliary outcomes are simply not well behaved enough to support the full specification.

### 2. Thin target-side post-treatment support amplifies the problem

The target-side diagnostics also confirm that post-treatment support is materially thinner for several of the failed outcomes, especially the arrival-based relative measures. In a number of target specifications, the usable post-treated sample is small relative to the corresponding acquiror sample. This is exactly the environment in which ratio-type outcomes become least stable: a few observations can dominate the upper tail, and the fixed-effects regression has less effective variation from which to identify the coefficient of interest.

So the right notebook interpretation is not that “target effects are absent because target regressions fail.” The better interpretation is narrower: **some target-side auxiliary outcomes are too thin and too distributionally unstable to estimate reliably in the full stacked specification**.

### 3. The new collinearity diagnostics also rule out the simplest coding-error story

The added collinearity summary is useful because it shows that, for the failed outcomes:
- the raw regressor matrix has full column rank,
- the controls are not constant,
- there are no obvious exact duplicate columns,
- and there is still substantial within-firm and cross-sectional variation in the lagged controls.

That is important. It means the failures are **not** explained by a trivial coding mistake such as accidentally duplicating a regressor, including an all-zero control, or losing all variation in the raw design matrix.

Instead, the remaining explanation is more subtle and more plausible in this setting: the rank failure is arising **after the interaction of outcome-specific sample selection with entity and time fixed effects**. In other words, once the regression is estimated on the exact non-missing sample for a given fragile outcome and the fixed effects absorb part of the variation, the transformed design can become singular even though the raw matrix is not.

### 4. Why this is not a threat to the core empirical message

These failures do not undermine the main results because they are concentrated in the least stable firm-level auxiliary outcomes rather than the core architecture of the project. The headline patterns of the notebook do **not** depend on being able to estimate every relative inventor-flow quality ratio. The strongest conclusions instead come from:
- the matched stacked design itself,
- the core patent and citation outcomes,
- the heterogeneity analyses,
- and the inventor-year mechanism evidence.

So the right way to present this in the notebook is transparent but not alarmist: some firm-level auxiliary outcomes are too thin or too irregularly distributed to support the full fixed-effects specification, especially on the target side. That is a limitation of those variables, not a reason to doubt the broader design.

### 5. Practical implication for the public-facing write-up

For a public notebook or GitHub presentation, I would explicitly note that:
- the `avg_rel_*` arrival and departure measures are informative as descriptive mechanism variables,
- but they should not be treated as the central empirical headline,
- and non-estimability for a subset of these outcomes is itself consistent with the fact that inventor-flow ratios are sparse, denominator-sensitive, and highly skewed in annual firm-level data.

That framing strengthens the project rather than weakening it, because it shows that the analysis does not force fragile outcomes into the headline story when the data do not support that level of precision.



## What the Sun–Abraham and CSDID files add

The additional SA and CSDID-related files are useful for evaluating credibility rather than just producing another coefficient.

### Sun–Abraham: some outcomes look cleaner than others
The acquiror SA files show an informative contrast.

- For **cited patents**, the pre-treatment coefficients at event times `-4`, `-3`, and `-2` are positive and non-trivial before the omitted period, while the post-treatment estimates drift toward zero or slightly negative values. That pattern suggests caution: part of the apparent post-merger decline may reflect a reversal from a stronger pre-treatment path rather than a clean break exactly at treatment.
- For **top10_to_2_patents**, pre-trends look more subdued, and the post-treatment estimates gradually become more negative, though modestly so.
- For **uncited patents**, pre-period coefficients are close to zero, while post-treatment coefficients become clearly negative from around event time `2` onward. This is the sort of pattern that is more naturally read as a dynamic post-treatment deterioration.
- For **self-cites**, pre-period coefficients are fairly flat, and the later post-treatment years turn negative. That is again suggestive of a delayed integration effect rather than an immediate jump at closing.

So SA does not only “confirm or reject” the baseline. It helps distinguish between:
- outcomes that may reflect underlying pre-treatment differences in trajectory,
- and outcomes where deterioration seems to emerge only after the merger.

### CSDID feasibility: identification support exists for the selected acquiror outcomes
The CSDID feasibility summaries are reassuring in one narrow but useful sense. For the selected acquiror outcomes, all cohort-time cells are identified in the feasibility files, with full cell coverage and non-trivial treated and control counts. That does not by itself prove the eventual causal estimate is clean, but it does show that the staggered-adoption design has adequate support for these outcomes rather than collapsing because of empty cohort-time cells.

That is worth mentioning in the notebook because it shows attention to design quality, not just point estimates.


## What the placebo files add now that we have baseline, Sun–Abraham, and BJS placebo outputs

The placebo block is now rich enough to support a much sharper credibility discussion than before. The right conclusion is **not** that the design fails. The right conclusion is that the placebo evidence is **asymmetric across roles and across estimators**, and that this asymmetry helps identify which findings should be treated as headline material.

### 1. Baseline placebo evidence: caution for pooled firm-level significance

The baseline pooled placebo regressions already suggest that this empirical environment can generate non-trivial pseudo-effects. The uploaded summary files show statistically significant placebo coefficients for `log1p_xi_real` in the acquiror permutation and lead exercises, and in the target permutation exercise, while the target lead placebo is close to zero. That pattern is cautionary rather than fatal. It implies that, in a short staggered event-study window with noisy innovation outcomes, a significant pooled TWFE coefficient is **not sufficient on its own** to claim a clean causal effect.

That is exactly why the notebook should keep the emphasis on:
- sign stability across related outcomes,
- event-study shape rather than one pooled estimate,
- agreement across estimators where available,
- and economically interpretable heterogeneity.

### 2. Sun–Abraham placebo evidence: cleaner on the target side than on the acquiror side

The Sun–Abraham placebo files are especially informative because they separate residual timing concerns from plain TWFE weighting concerns.

For the **target** placebo exercises, the SA files are broadly reassuring. The target placebo paths for total patents, num inventors, and cites are generally small relative to their standard errors in both the permutation and lead variants. That does **not** prove the target design is perfect, but it does suggest that once timing heterogeneity is handled more carefully, the target-side placebo evidence looks materially cleaner.

For the **acquiror** placebo exercises, the picture is weaker. The SA permutation placebos for total patents and num inventors are fairly muted, but several lead-placebo files remain noticeably positive, especially for total patents, num inventors, and arriving inventors. Departing inventors also show non-trivial pseudo-movement in the permutation exercise. The most honest reading is that the acquiror side still contains some residual dynamic structure or pseudo-treatment alignment that does not disappear completely under SA.

That matters for interpretation. It suggests that acquiror-side pooled firm outcomes should be framed more as **organized evidence of reallocation and heterogeneity** than as one clean average causal break.

### 3. BJS placebo evidence: pseudo-effects remain non-trivial in levels

The BJS placebo outputs do not include the same standard-error structure as the SA files, so they are best read descriptively rather than as a formal significance test. Even so, they are informative. The placebo `att_k` paths are often not close to zero in levels, especially for total patents and num inventors on both the acquiror and target sides. Lead-placebo BJS paths are also clearly non-flat.

That pushes in the same direction as the baseline and SA placebo evidence: the data contain enough dynamic structure that one should be wary of over-interpreting any single average patent-count coefficient.

### 4. Why this strengthens rather than weakens the public-facing notebook

Taken together, the placebo files do **not** tell us that nothing is there. They tell us to be more disciplined about **what** is there.

The most defensible claims after the placebo review are:

1. the project is strongest when it emphasizes **heterogeneity** rather than one pooled average effect,
2. the **target side** often looks cleaner than the acquiror side in heterogeneity-robust placebo checks,
3. the **acquiror side** still contains meaningful evidence of post-merger organizational adjustment, but those results should be framed as mechanism- and composition-oriented rather than as a single clean average decline or increase,
4. and placebo sensitivity is exactly why the notebook should foreground agreement across estimators and economic interpretation rather than isolated significance stars.

So the placebo evidence is best described as **cautionary but constructive**. It narrows the set of results that should be headlined, and in doing so it actually makes the final story more credible.


## Inventor-year baseline results: what the newly uploaded files add beyond the firm panel

The inventor-year panel now has enough supporting files to describe its empirical role more concretely. This branch is not just a mechanism appendix. It is the part of the project that distinguishes **organization-level disruption** from **inventor-level adjustment**.

The panel-overview files show that the inventor-year samples are large and balanced enough to be informative. The acquiror inventor-year panel contains about **4.11 million inventor-year-event rows**, around **119,682 inventors**, and a balanced event window from `-5` to `+5`. The target inventor-year panel contains about **2.01 million rows**, around **54,729 inventors**, and the same balanced event window. That scale is a strength of the notebook: the inventor-side results are not anecdotes from a tiny matched sample.

### 1. The pre-period means matter for interpretation

The pre-period baseline-mean files also show why the inventor-year results must be interpreted as **differential changes**, not as simple treated-control level comparisons.

For **acquirors**, treated inventors start above controls in pre-period patents, citations, `xi_real`, and move probability, but below controls in exploration. For **targets**, treated inventors also begin with higher pre-period citations, `xi_real`, exploration, and move probability than controls. This is not a design failure; it is a reminder that matching improves comparability without making treated and control inventors identical in levels. The identifying content therefore comes from the post-event path relative to those pre-existing differences.

### 2. Acquiror inventor-year baseline results: reallocation is clearer than raw productivity gain

The newly uploaded baseline significance files sharpen the acquiror story.

In the **no-x, no-firm** baseline specification:
- `total_patents` is positive and significant,
- `exploration_inv` is positive and significant,
- `is_move_year` is positive and significant,
- `novelty_score_group` is mildly positive,
- `xi_real` is negative,
- and `cites` is not clearly different from zero.

That already points toward a reallocation story: more output quantity and more exploratory behavior, but not a clean quality-adjusted productivity gain.

Once inventor controls are added in the **x, no-firm** specification, the picture becomes more selective:
- `total_patents` is no longer distinguishable from zero,
- `exploration_inv`, `is_move_year`, and novelty remain positive,
- `cites` turns positive,
- and `xi_real` becomes strongly negative.

Economically, that is a useful pattern. It suggests that merger exposure at acquirors is not best described as “inventors produce more.” It is better described as “inventors appear to search more broadly, move more, and reorient their activity, while the quality-adjusted efficiency of that activity weakens.”

### 3. Target inventor-year baseline results: the clearest inventor-level deterioration is on patenting

The target baseline files look different.

In the **no-x, no-firm** specification:
- `total_patents` is negative and significant,
- `cites` is negative and significant,
- `is_move_year` rises,
- `exploration_inv` rises while `exploitation_inv` falls,
- `xi_real` is positive,
- and novelty is essentially flat.

That combination is consistent with disruption, but not with a single one-dimensional decline. Inventors at target firms appear to become more mobile and more exploratory, while patent counts and citations weaken.

In the **x, no-firm** target specification, the negative patent result survives and becomes larger in magnitude, `is_move_year` remains positive, and exploration/exploitation effects remain present, but citations and novelty weaken materially. That makes the target-side patent decline the most durable baseline inventor result in the currently uploaded files.

### 4. Why the inventor-year baseline results complement the firm panel

Read together with the firm panel, the inventor-year results imply that mergers do not merely change total firm patent counts. They also reshape **who searches broadly, who moves, and how individual inventor output is reallocated**.

That is a strong public-facing insight. The firm panel says there is disruption. The inventor-year panel says the disruption appears to work partly through:
- broader inventor search,
- higher mobility around the merger window,
- and a wedge between output quantity and quality-adjusted performance.

That mechanism interpretation is more valuable than trying to claim that every inventor outcome points in the same direction.


## Inventor-year advanced methods: what the newly uploaded SA and CSDID files add

The advanced inventor-year estimators matter because the baseline inventor event studies are visibly not flat in every pre-period. The purpose of SA and CSDID here is therefore not just to accumulate robustness tables. It is to ask **which inventor-side patterns survive methods that handle staggered timing differently**, and which patterns are too sensitive to estimator choice or support.

A useful way to read the current upload set is to separate the advanced estimators into two groups:

1. **Sun–Abraham (SA)** is informative about whether the baseline TWFE event-study signs survive a heterogeneity-robust reweighting of event time.
2. **CSDID** is informative about cohort-specific treatment effects, but in this run several inventor-year `x_firm` CSDID outputs look numerically unstable or implausibly sharp, especially on the target side and, for some outcomes, even on the acquiror side. I therefore treat CSDID here as a diagnostic complement rather than the decisive tie-breaker.

### 1. Acquiror no-x / no-firm results: the reallocation story survives SA

For **acquiror total patents**, the no-x baseline event study shows a broadly positive post path, especially from about `t = +2` onward, though the leads are not flat. The SA path leans the same way: post estimates are positive from the event year onward and become larger in the middle and later post years. That is enough to keep a **positive later-post patenting response** in the set of serious interpretations, while still being explicit that the timing is not pristine.

For **acquiror exploration**, baseline TWFE and SA point in the same broad direction. The post path is positive from the event onward and remains positive in later years. The main caveat is again that the pre-period is not perfectly flat. So the safest statement is not that exploration rises with clinical certainty, but that merger exposure appears to **reorient inventive search** in a way that baseline TWFE and SA both read as more exploratory.

For **`is_move_year`**, SA is especially useful because it clarifies the shape of the adjustment. The acquiror move-year effect is positive around the event and in the early post years, then fades and can turn slightly negative later. That supports a **transition-period reshuffling** interpretation rather than a permanent upward shift in annual mobility.

### 2. Acquiror richer-control (`x_nofirm`, `x_firm`) SA results: exploration and mobility are more robust than citations

The additional SA files with inventor controls reinforce the same hierarchy of credibility.

- **Exploration** stays positive in both `x_nofirm` and `x_firm` SA files.
- **Move-year** still looks like an event-centered spike that fades later.
- **Citations** remain noisy and are not a good headline result because the pre-period is not clean and the post path is uneven.

That combination is informative. Once richer controls are added, the acquiror inventor story still looks more like **reallocation, search reorientation, and temporary mobility** than like an across-the-board inventor-quality improvement.

### 3. Target SA results: the negative patenting story survives, especially in later post years

For **targets**, the SA evidence is most persuasive for patent quantity.

In the no-x specification, target inventor-year patents are negative after the event, with the deterioration becoming more pronounced in later post years. In the `x_firm` specification, the same general message remains, although the path is noisier and the pre-period is not flat. The most negative coefficients arrive in the later post period, especially around `t = +4` and `t = +5`.

For **target citations**, the SA file is directionally consistent with deterioration but too noisy to use as a clean stand-alone headline. The pre-period includes sizable negative estimates, the early post period is mixed, and only the late post period turns clearly negative. So citations are best presented as **supportive but weaker** evidence relative to target patent counts.

### 4. What to do with the CSDID files

The CSDID uploads add information, but they should be handled carefully.

For the **acquiror `x_firm` outcomes**, the CSDID signs are broadly intuitive for some outcomes:
- `total_patents` is positive on average after treatment,
- `is_move_year` spikes at `e = 0` and then turns negative in later post periods,
- `exploration_inv` is positive at impact but negative in later post periods,
- and `cites` is very strongly positive.

Those sign patterns are not absurd economically. They are broadly consistent with a story in which the event year features immediate reshuffling and search expansion, while later post averages depend on cohort composition and selective persistence. But the **standard errors are extremely small**, especially relative to the scale of the outcome, so I would not treat the acquiror `x_firm` CSDID files as the cleanest quantitative robustness check.

For the **target `x_firm` outcomes**, the concern is stronger. The newly uploaded dynamic and `attgt` files still show effectively zero standard errors in many cells and mechanically sharp confidence intervals. That means they are best described as **diagnostic output from this run, not as a reliability-enhancing robustness layer**.

### 5. Bottom line on advanced inventor-year methods

The most defensible synthesis is this:

- **Acquiror inventors:** SA supports a reallocation story with later patent gains, more exploratory search, and a short-run mobility spike. CSDID does not overturn that interpretation, but its `x_firm` outputs are not stable enough in this run to deserve primary weight.
- **Target inventors:** SA supports the view that post-merger inventor patenting weakens, especially in later post years. Target citations lean the same way but are less clean. Target `x_firm` CSDID is too degenerate in this run to be a major robustness pillar.

So the advanced-method section strengthens the notebook, but not by saying every estimator lines up perfectly. It strengthens the notebook by showing that **the acquiror side looks like reorganization and search reorientation, whereas the target side looks more like inventor-level productivity deterioration**, and that this broad split survives the most useful advanced estimator currently available in the run, namely Sun–Abraham.


## Additional discussion: inventor-year event studies after the latest uploads

The inventor-year event-study evidence is now rich enough to compare baseline TWFE dynamics directly with the advanced estimators for the most relevant acquiror and target outcomes.

### Acquiror inventors: later patent gains, immediate search reorientation, and transition-period mobility

For **total patents**, the baseline acquiror event study turns clearly positive in later post years, especially around `t = +2`, `+3`, and `+5`. The no-x Sun–Abraham results lean the same way: the post-treatment path is positive and strongest in the middle and later post years. This is one of the few inventor-year outcomes for which several currently uploaded estimators point in the same broad direction. The caveat is that the pre-period is not flat, so the correct interpretation is **later-post strengthening**, not a perfectly clean discrete break at the merger date.

For **exploration**, the message is clearer after the newest uploads. The baseline event study is positive in the post period, the SA path is also positive after the event in the no-x, `x_nofirm`, and `x_firm` specifications, and the effect remains economically interpretable even when the exact magnitude varies. That makes exploration one of the more convincing **behavioral** acquiror outcomes, even if it is not the cleanest purely causal timing result.

For **cites**, the baseline event study is still not a good headline result. The post-treatment path becomes positive in later years, but the leads are visibly non-flat. The inventor-control SA file also remains uneven. So citations are better treated as **suggestive and secondary**.

For **`is_move_year`**, the combined baseline and SA evidence now supports a fairly specific shape: mobility is elevated around the event and in early post years, but that increase fades later and can reverse. That is exactly what one would expect if the merger induces a temporary wave of reassignment, matching, or separation rather than a permanently higher annual move propensity.

### Target inventors: later post deterioration rather than immediate collapse

For **target total patents**, the baseline and SA event studies tell a similar qualitative story. The post path is not sharply negative on impact, but it drifts downward and becomes more negative in later post years. This matters economically because it fits an integration-friction or resource-withdrawal story better than a pure “shock at closing” story. In other words, target inventors do not necessarily stop patenting immediately; instead, deterioration appears to accumulate as the integration process unfolds.

For **target citations**, the currently uploaded SA file is noisier than the patent-count evidence and has problematic leads. It still points directionally toward deterioration in the later post years, but it is better used as a **corroborating outcome** than as a stand-alone centerpiece.

### How these event-study patterns should be presented

The strongest way to present the dynamic inventor evidence is not to insist that every pre-period is flat. It is to emphasize three things:

1. **Acquiror inventor dynamics look like reallocation.**  
   Patenting strengthens later, exploratory behavior rises, and mobility spikes around the event before fading.

2. **Target inventor dynamics look like deterioration.**  
   Patent output weakens most clearly in later post years, with citations moving in the same direction but less cleanly.

3. **Dynamic timing is informative but not immaculate.**  
   The notebook should be explicit that these results are informative because the signs and shapes are economically coherent across several files, not because every lead coefficient is ideally close to zero.


## Additional discussion: inventor-year triple-DiD by inventor standing within the firm

The newly uploaded inventor-year triple-DiD result for the acquiror no-x, no-firm specification is especially useful because it makes the mechanism story more concrete.

The key specification splits inventors by whether they are in the **upper half of the within-firm cumulative patent distribution**. That is a simple but meaningful proxy for an inventor's standing inside the firm's internal innovation hierarchy. Inventors above that threshold are more established contributors to the firm's historical patent stock; inventors below it are more peripheral, junior, or newly important relative to the firm's older core.

### What the uploaded coefficient says

In the uploaded total-patent triple-DiD file:
- the baseline `Post_Treated` effect is strongly positive,
- the `Post_Z_upper_half_cum_patents_within_firm` term is strongly negative,
- and the triple interaction `Post_Treated_Z_upper_half_cum_patents_within_firm` is also strongly negative.

The economic implication is straightforward: the positive post-merger patent response is **substantially weaker for inventors in the upper half of the within-firm cumulative patent distribution** than for the omitted lower-half group.

### Why that matters economically

This is an important refinement of the inventor-side story. It suggests that the merger response is not driven primarily by the firm's most established incumbent inventors. Instead, the action appears more concentrated among inventors who are less central in the firm's pre-merger patent hierarchy.

That pattern fits a reallocation story well:

- core inventors may be tied to mature projects, stable teams, and established routines,
- lower-ranked inventors may be more movable across teams and technology combinations,
- and post-merger integration may create more scope for those lower-ranked inventors to pick up new patenting opportunities.

In other words, the merger does not simply “boost inventor output.” It appears to **redistribute where inside the organization the inventive response shows up**.

### Why this is a strong result for the notebook

This is exactly the kind of finding that makes the project more than a standard DiD exercise. It connects the average treatment discussion to an organizational mechanism:

- not every firm responds the same way,
- not every inventor responds the same way,
- and the merger shock appears to work through internal rank, assignment, and reallocation inside the innovation workforce.

That heterogeneity result is more interesting than another average patent coefficient, and it is well aligned with the broader theme of the notebook.



## Why the heterogeneity story remains the most credible result

The inconsistency in some average treatment effects does **not** make the project weak. If anything, it makes the stronger heterogeneity evidence more valuable.

Why?

1. **Average effects are clearly not uniform.**  
   That is already an important empirical finding in a setting where theory would not predict uniform adjustment across all firms.

2. **The moderation pattern is systematic rather than ad hoc.**  
   Acquiror results repeatedly emphasize firm size and capacity; target results repeatedly emphasize relative deal scale.

3. **The machine-learning results align with the parametric design.**  
   In the causal-forest summaries uploaded so far, `lag1_log_sale` is repeatedly the most important or one of the most important heterogeneity drivers for acquiror outcomes, and it also ranks highly for target outcomes. That is exactly what one would hope to see if the parametric triple-DiD is capturing a real moderator rather than data mining noise.

4. **The mechanism interpretation is compelling.**  
   Mergers do not only combine assets; they combine decision rights, reporting structures, project portfolios, and inventor teams. A story centered on absorptive capacity and transaction intensity is therefore more plausible than a one-size-fits-all average effect.

That is why the recruiting-oriented presentation should lean into the idea that this project studies **which firms absorb innovation shocks better and why**, not merely whether one average coefficient is above or below zero.


## Recommended main story for the notebook and GitHub project

I would recommend the following narrative.

### Main claim
**M&A does not generate one uniform innovation effect. Instead, the evidence points to post-merger disruption and reallocation whose consequences depend on organizational capacity, deal context, and inventor position inside the firm.**

### Supporting interpretation
- On average, pooled baseline DiD and event-study estimates often show weaker post-deal innovation outcomes, but those average effects are uneven across outcomes and sometimes complicated by pre-trend and placebo concerns.
- For the **firm panel**, the most credible result remains **heterogeneity** rather than a single average treatment effect. Baseline sales is the clearest acquiror-side moderator, while deal intensity relative to the target is the clearest target-side moderator in the broader project evidence.
- The new placebo files reinforce that message. Baseline significance alone is not enough in this setting. The target side looks cleaner than the acquiror side under Sun–Abraham placebo checks, while BJS placebo paths are often non-trivial in levels for both roles.
- For the **inventor-year panel**, the strongest message is that merger exposure changes **how inventors work and where adjustment occurs**, not just whether the average inventor patents more or less.
- On the **acquiror inventor** side, the evidence is most consistent with reorientation: later patenting strength, changed search behavior, a transition-period mobility spike, and weaker quality-adjusted performance in some specifications.
- On the **target inventor** side, the clearest currently uploaded result is a decline in patenting, alongside higher mobility and more exploratory behavior in baseline specifications.
- The within-firm-rank triple-DiD adds a particularly strong mechanism result: post-merger patent gains are weaker for inventors already high in the firm's cumulative patent hierarchy, which points toward internal reallocation rather than a uniform lift to the incumbent core.

### Why this is a strong recruiting story

This is the kind of empirical project a tech economist or applied microeconomist should want to show publicly:
- it builds a difficult linked inventor-firm-deal dataset,
- it separates organization-level outcomes from inventor-level mechanisms,
- it treats staggered timing seriously,
- it checks placebo sensitivity rather than hiding it,
- it distinguishes average effects from heterogeneity,
- and it turns a noisy empirical setting into a disciplined statement about absorptive capacity, integration frictions, and internal reallocation of inventive effort.

One additional clarification remains important for the firm-panel results. A subset of the auxiliary mobility-ratio variables does not survive the full fixed-effects specification. The current diagnostics still suggest that this is best interpreted as a support and variable-construction issue for sparse denominator-driven outcomes, not as a general design failure. Those variables should remain in the notebook as secondary mechanism evidence rather than as headline results.


## Draft text for the notebook's expanded main results section

**Expanded draft results discussion**

The most credible way to summarize the project is not to say that mergers simply help or hurt innovation. The evidence is more informative than that. Across the firm and inventor panels, the results point to **post-merger disruption and reallocation**, with the sign and magnitude of the response depending on where one looks: at the firm as a whole, at inventors inside the organization, across larger versus smaller firms, and across more central versus more peripheral inventors.

At the **firm level**, the baseline matched TWFE results still provide the useful first pass: targets often look weaker after the deal, while acquirors show more mixed average responses. But the accumulated placebo evidence makes clear that one significant pooled coefficient should not be treated as decisive in this design. Baseline placebo regressions can generate pseudo-effects, Sun–Abraham placebo checks look cleaner on the target side than on the acquiror side, and BJS placebo paths are often not close to zero in levels. The right lesson is not that the firm-level analysis collapses. The lesson is that the strongest firm-level contribution of the project is the **heterogeneity structure**, not one universal average effect size.

That heterogeneity story remains compelling. The broader notebook evidence continues to suggest that larger and better-positioned acquirors are more capable of absorbing merger-related disruption, while targets exposed to more intense or more consequential deals experience sharper deterioration. This is a richer and more credible conclusion than a generic “M&A lowers innovation” claim because it links the empirical patterns to organizational absorptive capacity and integration frictions.

The **inventor-year panel** adds the mechanism layer. The panel-overview files show that these samples are large and balanced enough to support serious interpretation, but the pre-period means also make clear that treated inventors are not level-identical to their matched controls. That means the core identifying content lies in differential changes over time, not in treated-control comparability in levels.

Within that inventor-year layer, the acquiror and target stories now separate fairly clearly. For **acquiror inventors**, the strongest average and dynamic evidence points to **reallocation** rather than broad productivity improvement. In the broader no-x and x specifications, patenting often strengthens later in the post period, exploration rises, and move probability spikes around the event and then fades. But once both inventor and firm controls are imposed, broad patent and citation gains are no longer robust average effects. What remains most durable is the idea that merger exposure changes **how acquiror inventors work**: they search differently, move differently, and do not uniformly improve on quality-adjusted performance.

For **target inventors**, the strongest evidence is more clearly negative. In both baseline and Sun–Abraham results, target-side inventor patenting weakens after the merger, especially in later post years. In the richer `x_firm` baseline, citations also fall. That makes the target inventor story less about a stable behavioral reorientation and more about **inventor-level productivity deterioration during integration**.

The heterogeneity files make that mechanism story much more persuasive. On the acquiror side, inventor responses vary systematically with firm size and with the inventor's pre-merger standing inside the firm. Larger acquirors are better able to translate merger exposure into inventor-side patent and citation gains, while smaller acquirors appear to experience more disruptive reshuffling without the same payoff. At the same time, post-merger patent gains are weaker for inventors already high in the firm's cumulative patent hierarchy, which suggests that lower-ranked inventors play an important role in the reallocation margin. On the target side, the comparable heterogeneity results are better read as identifying **who deteriorates more sharply**, especially among more central inventors and in less favorable firm contexts.

The advanced inventor-year estimators help, but asymmetrically. Sun–Abraham is genuinely useful in this run because it preserves the broad split between acquiror reallocation and target deterioration while handling staggered timing more carefully. The inventor-year CSDID files are more mixed. Some acquiror `x_firm` outputs are directionally interpretable but come with suspiciously tiny standard errors, while several target `x_firm` outputs are numerically degenerate. That means the notebook should use CSDID as **diagnostic support**, not as the decisive robustness layer.

Taken together, the project's strongest contribution is therefore not a claim that mergers have one average effect on innovation. It is a more interesting organizational conclusion: **mergers reshape innovation through disruption, reallocation, and uneven absorptive capacity**, and those effects are visible both at the firm level and inside the inventor workforce.


In [ ]:
# Optional convenience cell:
# once the heavy analysis has been run and outputs exist, use the helper file
# to create notebook-ready support CSVs in one step.
#
# support = write_notebook_support_outputs(
#     output_root=MODEL_OUT,
#     preferred_outcomes=[
#         "is_move_year",
#         "exploration_inv",
#         "arriving_inventors_count",
#         "departing_inventors_count",
#         "total_patents",
#         "cites",
#         "xi_real",
#     ],
# )
# support

## Final update from batches 6 and 7 plus the remaining CSDID files: inventor-year `x_firm` results are informative, but the advanced-method layer is asymmetric in quality

The final uploaded batches materially improve the inventor-year discussion because they add the missing `x_firm` heterogeneity and CSDID files. The notebook can now separate fairly cleanly between (i) results that deserve headline weight, (ii) results that help as mechanism evidence, and (iii) results that are too numerically fragile in this run to carry much weight.

### 1. What survives in the `x_firm` baseline specifications

The `x_firm` baseline significance files sharpen the split between acquirors and targets.

For **acquirors**, once inventor and firm controls are included:
- `total_patents` is slightly negative and not statistically different from zero,
- `cites` is also not significant,
- but `exploration_inv`, `is_move_year`, and `novelty_score_group` remain positive and precisely estimated,
- while `xi_real` remains sharply negative and `exploitation_inv` moves in the opposite direction to exploration.

This is an important result because it narrows the acquiror inventor story. After rich controls, the cleanest remaining average effects are **search reorientation, mobility/reshuffling, and weaker quality-adjusted efficiency**, not broad inventor-level output gains.

For **targets**, the `x_firm` baseline remains more clearly productivity-oriented:
- `total_patents` is negative and significant,
- `cites` is also negative and significant,
- while exploration, exploitation, novelty, and `xi_real` are mostly weak or unstable in the same richer-control sample.

That makes the target-side inventor story more focused: once the sample is restricted to observations with richer firm-level support, the most durable effect is **productivity deterioration**, not a broad and stable behavioral reorientation.

### 2. What the new SA files add

The new SA files reinforce that same distinction.

For **acquirors**, the `x_firm` SA files show:
- exploration is positive after the event and remains positive through the post period,
- `is_move_year` rises at impact and in early post years, then fades and turns negative later.

So even when patents and citations are not robust average winners in the rich-control baseline, the dynamic SA evidence still says that the merger changes **how inventors search and how they move**.

For **targets**, the `x_firm` SA patent file is especially informative:
- the path is mixed or even slightly positive on impact and at `t = +1`,
- but it turns sharply negative in later post years, especially around `t = +4` and `t = +5`.

The target `x_firm` citation SA file is noisier, but it leans in the same late-negative direction. So the best dynamic reading is that target inventor deterioration is **progressive rather than instantaneous**.

### 3. What the remaining CSDID files add — and why they still need caution

The remaining CSDID uploads complete the coverage, but they do not all increase confidence equally.

For **acquiror `x_firm` CSDID**, the files do add economically interpretable sign patterns:
- `total_patents` is positive on average in post periods, though slightly negative at `e = 0`,
- `is_move_year` spikes positively on impact and turns negative afterward,
- `exploration_inv` is positive at impact but negative in later post periods,
- `cites` is strongly positive throughout.

These patterns fit a coherent story in which the event year features immediate organizational disruption and selective reallocation, with later-post averages reflecting which inventors and projects persist. Still, the standard errors are extremely small, so I would use the acquiror `x_firm` CSDID results as **directional support**, not as the quantitatively cleanest robustness layer.

For **target `x_firm` CSDID**, the newly uploaded `dynamic` and `attgt` files confirm rather than resolve the earlier concern: many cells still have effectively zero standard errors and mechanically exact confidence intervals. That makes the target `x_firm` CSDID block unsuitable as a strong robustness claim in the current run.

### 4. What deserves headline weight after all batches

After all uploads, the `x_firm` inventor-year evidence supports three headline statements:

1. **Acquiror inventors:** the most robust `x_firm` effects are reallocation-style outcomes — exploration, mobility timing, novelty, and weaker `xi_real` — not broad patent or citation gains.

2. **Target inventors:** the most robust `x_firm` effects are negative productivity outcomes — lower patenting and lower citations — with the strongest deterioration appearing in later post years.

3. **Advanced-method asymmetry is itself informative:** SA is the most useful advanced estimator in the current run for inventor-year interpretation, while the target-side `x_firm` CSDID outputs remain too numerically degenerate to serve as decisive evidence.


## Final update on inventor-year heterogeneity: firm size and inventor rank look like mechanism variables, not decorative moderators

The new `x_firm` triple-DiD files deepen the mechanism interpretation in a way that is highly useful for the notebook's target audience.

### 1. Acquirors: firm scale systematically moderates inventor responses

The acquiror `x_firm` heterogeneity files using `Z_log_sales_*` show a structured pattern rather than random scatter.

Across the continuous and quantile-based sales splits:

- the baseline `Post_Treated` effect for `total_patents` is often weak or negative for the omitted smaller-size group,
- but the triple interactions for larger sales groups are frequently positive and significant, especially in the upper terciles and upper quintiles,
- citation effects also become much more favorable in larger sales groups,
- while novelty and, in some bins, mobility become less favorable in the largest groups,
- and the exploration response is strongest in some middle groups but attenuates or reverses in the largest groups.

That combination is economically intuitive. Larger acquirors appear better able to convert merger exposure into inventor-side output and citation performance, whereas smaller acquirors look more exposed to integration frictions and less able to translate organizational change into immediate inventive gains. At the same time, the strongest exploration and movement responses are not always concentrated in the very largest groups, which fits the idea that smaller or mid-sized acquirors may experience more disruptive reshuffling even if they do not convert that reshuffling into better output.

### 2. Acquirors: inventor rank inside the firm remains one of the strongest mechanism results

The `Z_upper_half_cum_patents_within_firm` files are among the most interesting inventor-level outputs in the entire notebook.

In the richer `x_firm` specification, the triple interaction is strongly negative for:
- `total_patents`,
- `cites`,
- `xi_real`,
- and `is_move_year`.

At the same time, the triple interaction is positive for `exploration_inv` and negative for `exploitation_inv`.

That pattern says the inventors who were already in the upper half of the firm's cumulative patent hierarchy do **not** drive the positive post-merger inventor response. If anything, their output, citation, and mobility responses are weaker. The lower-ranked group appears more responsive in patenting and movement, while the higher-ranked group tilts relatively more toward exploration. This is exactly the sort of mechanism result that makes the project look like organizational economics rather than just reduced-form accounting.

### 3. Targets: heterogeneity mostly tells us who is hit harder

The target `x_firm` heterogeneity results are less rich than the acquiror side, but they are still informative.

For the sales-based target splits:
- the continuous and quantile specifications often show negative triple interactions for `total_patents` and `cites`,
- especially for lower or middle sales groups,
- while most behavioral outcomes remain weak.

So on the target side, firm-scale heterogeneity is best interpreted as **moderation of deterioration** rather than as a positive reallocation channel. The more economically relevant question is not who benefits, but which target-side environments are associated with sharper post-merger decline.

The target within-firm-rank split tells a similar story. The triple interaction is strongly negative for `total_patents` and `cites`, which implies that the negative target-side productivity response is particularly strong for inventors who were already high in the firm's cumulative patent hierarchy. That is a meaningful and plausible result: central target inventors may be exactly the people whose projects, autonomy, or organizational fit are most disrupted by post-merger integration.

### 4. Why this heterogeneity block matters so much

The project becomes substantially stronger once these heterogeneity results are taken seriously.

Without them, the notebook would read as a mixed set of average effects with imperfect pre-trends and uneven placebo performance. With them, it reads as a structured organizational story:

- larger acquirors absorb and redeploy inventive talent better than smaller ones,
- lower-ranked acquiror inventors appear especially important for post-merger patent expansion,
- and high-rank target inventors appear especially exposed to deterioration after integration.

That is a much more interesting conclusion for PhD tech economists and recruiters than a yes/no claim about whether mergers are “good” or “bad” for innovation.


## Final completeness note after the remaining uploads

At this point, the uploaded result set is sufficient to support a rounded-out notebook discussion. I do **not** see a major missing result family that is necessary to finish the interpretation.

Two clarifications are worth recording explicitly:

1. The absent inventor-year SA files for some `x_firm` outcomes are **not an omission in my reading of the folder**; they were simply **not produced in this run**. That means the advanced-method discussion should be written asymmetrically and honestly, rather than pretending we have fully symmetric SA coverage for every outcome and role.

2. The remaining target-side `x_firm` CSDID files do not solve the earlier numerical concern. The dynamic and `attgt` outputs still show effectively zero standard errors in many cells, so they should remain in the notebook as diagnostic or provisional evidence, not as a strong robustness pillar.

Given the files now uploaded, I would regard the notebook as empirically complete enough for its intended purpose:

- the **firm panel** has baseline, heterogeneity, advanced estimators, and placebo evidence,
- the **inventor-year panel** now has baseline, dynamic, SA, and heterogeneity evidence for the key outcomes,
- and the **`x_firm` layer** is strong enough to separate a main story from a robustness layer even though some advanced-method coverage is incomplete or numerically fragile.

So the main remaining task is not to wait for more files. It is to present the current evidence clearly: emphasize the structured heterogeneity, be candid about timing and placebo limitations, and treat the most fragile advanced outputs with the right amount of caution.
